# 🏭 Industry 4.0 — Offline Visual Quality Inspector (ML Notebook)

**This self-contained notebook includes both codebases and the ML workflow:**
1. `dataset_manager.py` — dataset registry, download, folder organization, summary and MVTec loader utilities.
2. `industry4_ml_utils.py` — dataset adapters, validation, Pascal-VOC conversion, confidence calibration, drift perturbations and tiled inference.
3. Main notebook — training/evaluation for **YOLO defect localization** or **MobileNet image classification**, plus an optional PatchCore-inspired anomaly branch.

### Recommended first run
- Default dataset: **GC10-DET from Hugging Face** (`gc10_hf`) — a defect-localization dataset with box labels and train/validation/test splits.
- GPU: Runtime → Change runtime type → T4 GPU (if available).
- Run cells in order. The notebook does **not** fabricate labels, boxes, metrics, severity, or field-accuracy claims.

**Data warning:** NEU Surface Defect Database is commonly used as a six-class image-level classification dataset. The notebook auto-routes it to classification unless it finds genuine YOLO/VOC annotations. Public benchmarks are not a substitute for production batch-held-out data.

## 1. Install dependencies

Colab already includes PyTorch in most runtimes. These installs add the dataset, model, and export tools. Restart the runtime only if Colab explicitly asks you to.

In [ ]:
%pip -q install "ultralytics>=8.3,<9" "kagglehub>=0.3" "kaggle>=1.7" "huggingface_hub>=0.26" "scikit-learn>=1.4" "onnx>=1.16" "onnxruntime>=1.20" "opencv-python-headless>=4.8" "PyYAML>=6" "pandas>=2.0" "matplotlib>=3.7"

## 2. Configuration and reproducibility

Change `DATA_MODE` only when you want another dataset. For a custom annotated dataset, set `DATA_MODE='custom_yolo'` and provide its `data.yaml`. `ANOMALY_MODE` is off by default; enable it only with a normal/defect set from the **same product domain**.

In [ ]:
import os, sys, gc, json, random, shutil, time, math, warnings, zipfile, hashlib
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import cv2
import yaml
import torch
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from IPython.display import display

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
ROOT = Path('/content/industry4_qc')
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)

# Dataset-manager choices: neu, pcb, pcb_small, magnetic_tile, gc10, gc10_hf
# Also: custom_yolo (requires a user-supplied YOLO data.yaml)
DATA_MODE = 'gc10_hf'
CUSTOM_YOLO_YAML = Path('/content/drive/MyDrive/my_factory_dataset/data.yaml')

MODEL_NAME = 'yolo11n.pt'      # Edge-friendly baseline; compare yolo11s.pt later
EPOCHS = 50
IMGSZ = 640                   # Compare 640 vs 960 for tiny defects
BATCH_SIZE = 16 if torch.cuda.is_available() else 4
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
WORKERS = 2 if torch.cuda.is_available() else 0
RUN_TRAINING = True
RUN_EXPORT = True

# Optional anomaly branch: 'none', 'mvtec_manual', or 'custom'
ANOMALY_MODE = 'none'
MVTEC_ROOT = ROOT / 'data' / 'raw' / 'mvtec'  # extracted MVTec root, category folders inside
MVTEC_CATEGORY = 'tile'
CUSTOM_ANOMALY_ROOT = ROOT / 'custom_anomaly_data'
# Custom anomaly layout: train/normal, val/normal, test/good, test/defect
ANOMALY_PRODUCT_ID = 'custom_product'  # match this to the detector domain before fusion
ANOMALY_MEMORY_MAX = 1200
ANOMALY_CALIBRATION_QUANTILE = 0.99

print('Torch:', torch.__version__, '| Device:', DEVICE, '| Project root:', ROOT)

## 3. Code module #1 — Dataset Manager

This cell writes the dataset manager into the Colab working directory. Its registry matches your uploaded code; the utilities have safer errors and a reproducible classification split.

In [ ]:
%%writefile /content/industry4_qc/dataset_manager.py
#!/usr/bin/env python3
"""Dataset manager for the Industry 4.0 offline visual quality inspector.

Preserves the user's dataset registry while providing safer download, image-folder
organization, dataset inspection, and MVTec-style anomaly loading utilities.
"""
from __future__ import annotations

import argparse
import random
import shutil
import subprocess
import zipfile
from collections import Counter
from pathlib import Path
from typing import Optional


IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
DATASETS = {
    "neu": {"source": "kaggle", "id": "kaustubhdikshit/neu-surface-defect-database", "desc": "NEU steel surface defect dataset; commonly used for image-level classification, 1,800 images."},
    "pcb": {"source": "kaggle", "id": "norbertelter/pcb-defect-dataset", "desc": "PCB defect dataset (annotation format must be checked before use)."},
    "pcb_small": {"source": "kaggle", "id": "akhatova/pcb-defects", "desc": "Small PCB defect dataset (annotation format must be checked before use)."},
    "magnetic_tile": {"source": "kaggle", "id": "alex000kim/magnetic-tile-surface-defects", "desc": "Magnetic tile surface defects; usually used for classification/anomaly experiments."},
    "gc10": {"source": "kaggle", "id": "alex000kim/gc10det", "desc": "GC10-DET steel defect detection dataset."},
    "gc10_hf": {"source": "hf", "id": "dronefreak/GC10-DET", "desc": "GC10-DET steel defect detection dataset with YOLO-style files and splits."},
    "severstal_hf": {"source": "hf", "id": "Voxel51/severstal_steel_defects", "desc": "Severstal steel defect dataset; segmentation task, not automatically converted to boxes."},
    "mvtec_hf": {"source": "hf", "id": "Voxel51/mvtec-ad", "desc": "MVTec AD mirror; verify layout and licensing."},
    "adnet": {"source": "hf", "id": "linglingling009/ADNet", "desc": "Large anomaly dataset; download only when storage is sufficient."},
}
HUGE_DATASETS = {"adnet"}
MANUAL_DATASETS = {
    "MVTec AD official": "https://www.mvtec.com/company/research/datasets/mvtec-ad",
    "MVTec AD 2": "https://www.mvtec.com/company/research/datasets/mvtec-ad-2",
    "MVTec LOCO AD": "https://www.mvtec.com/company/research/datasets/mvtec-loco-ad",
    "MVTec 3D-AD": "https://www.mvtec.com/company/research/datasets/mvtec-3d-ad",
    "VisA AWS Open Data": "https://registry.opendata.aws/visa/",
    "KolektorSDD2": "https://www.vicos.si/resources/kolektorsdd2/",
    "Real-IAD": "https://realiad4ad.github.io/Real-IAD/",
    "Kaputt": "https://www.kaputt-dataset.com/",
    "NEU official": "https://faculty.neu.edu.cn/songkc/en/zdylm/263265/",
}


def is_image(path: Path) -> bool:
    return Path(path).suffix.lower() in IMAGE_EXTENSIONS



def list_datasets() -> None:
    print("Registered datasets:")
    for name, cfg in DATASETS.items():
        tag = " [LARGE]" if name in HUGE_DATASETS else ""
        print(f"\n{name}{tag}\n  source: {cfg['source']}\n  id: {cfg['id']}\n  description: {cfg['desc']}")


def print_manual_datasets() -> None:
    print("Manual/official dataset links:")
    for name, url in MANUAL_DATASETS.items():
        print(f"{name}: {url}")


def _has_images(path: Path) -> bool:
    return path.exists() and any(is_image(p) for p in path.rglob("*"))


def _copy_tree_contents(source: Path, destination: Path) -> None:
    source = Path(source)
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    if source.resolve() == destination.resolve():
        return
    for item in source.iterdir():
        target = destination / item.name
        if item.is_dir():
            shutil.copytree(item, target, dirs_exist_ok=True)
        else:
            shutil.copy2(item, target)


def download_dataset(name: str, root: Path = Path("data/raw"), force: bool = False) -> Path:
    """Download a registered dataset. Returns the destination directory.


    Kaggle uses kagglehub first and falls back to the Kaggle CLI. Hugging Face
    datasets use snapshot_download. Network and account access may be required.
    """
    if name not in DATASETS:
        raise KeyError(f"Unknown dataset {name!r}. Valid names: {sorted(DATASETS)}")
    destination = Path(root) / name
    destination.mkdir(parents=True, exist_ok=True)
    if _has_images(destination) and not force:
        print(f"Using existing images in: {destination}")
        return destination
    cfg = DATASETS[name]
    errors = []
    if cfg["source"] == "kaggle":
        try:
            import kagglehub
            downloaded = Path(kagglehub.dataset_download(cfg["id"]))
            _copy_tree_contents(downloaded, destination)
        except Exception as exc:
            errors.append(f"kagglehub: {exc}")
            try:
                cmd = ["kaggle", "datasets", "download", "-d", cfg["id"], "-p", str(destination), "--unzip"]
                subprocess.run(cmd, check=True, capture_output=True, text=True)
            except Exception as cli_exc:
                errors.append(f"Kaggle CLI: {cli_exc}")
    elif cfg["source"] == "hf":
        try:
            from huggingface_hub import snapshot_download
            snapshot_download(repo_id=cfg["id"], repo_type="dataset", local_dir=str(destination))
        except Exception as exc:
            errors.append(f"Hugging Face: {exc}")
    if not _has_images(destination):
        raise RuntimeError(
            f"Download failed or produced no image files for {name}.\n"
            f"Details: {' | '.join(errors) or 'No download result'}\n"
            "If Kaggle requires authentication, add kaggle.json in Colab or download manually."
        )
    print(f"Dataset ready at: {destination}")
    return destination



def summarize_folder(source: Path) -> dict:
    source = Path(source)
    if not source.exists():
        raise FileNotFoundError(source)
    counts = Counter()
    for image in source.rglob("*"):
        if image.is_file() and is_image(image):
            relative = image.relative_to(source)
            class_or_split = str(relative.parent) if len(relative.parts) > 1 else "root"
            counts[class_or_split] += 1
    print(f"Image summary: {source}")
    for name, count in sorted(counts.items()):
        print(f"{count:6d}  {name}")
    print(f"TOTAL: {sum(counts.values())}")
    return dict(counts)


def organize_classification_dataset(
    source: Path,
    destination: Path,
    seed: int = 42,
    train_ratio: float = 0.70,
    val_ratio: float = 0.15,
    clear_destination: bool = True,
) -> Path:
    """Create class-folder train/val/test splits from a flat class-folder source.

    Every class needs at least 3 images so all three splits can contain examples.
    This is a random image split; batch/group splitting is preferable for factory data.
    """
    source, destination = Path(source), Path(destination)
    if not source.exists():
        raise FileNotFoundError(source)
    if not (0 < train_ratio < 1 and 0 < val_ratio < 1 and train_ratio + val_ratio < 1):
        raise ValueError("train_ratio and val_ratio must be positive and sum to less than 1")
    class_dirs = sorted(
        d for d in source.iterdir() if d.is_dir() and any(is_image(f) for f in d.iterdir() if f.is_file())
    )
    if len(class_dirs) < 2:
        raise ValueError(f"Expected at least two class folders directly under {source}")
    if clear_destination and destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True, exist_ok=True)
    rng = random.Random(seed)
    test_ratio = 1.0 - train_ratio - val_ratio
    for class_dir in class_dirs:
        files = sorted(p for p in class_dir.iterdir() if p.is_file() and is_image(p))
        if len(files) < 3:
            raise ValueError(f"Class {class_dir.name} has {len(files)} images; need at least 3 for train/val/test")
        rng.shuffle(files)
        n_train = max(1, int(round(len(files) * train_ratio)))
        n_val = max(1, int(round(len(files) * val_ratio)))
        n_test = len(files) - n_train - n_val
        if n_test < 1:
            n_test = 1
            if n_train > 1:
                n_train -= 1
            else:
                n_val -= 1
        splits = {"train": files[:n_train], "val": files[n_train:n_train+n_val], "test": files[n_train+n_val:]}
        for split, split_files in splits.items():
            target_dir = destination / split / class_dir.name
            target_dir.mkdir(parents=True, exist_ok=True)
            for index, path in enumerate(split_files):
                shutil.copy2(path, target_dir / f"{index:06d}_{path.name}")
    print(f"Prepared class-folder split: {destination}")
    return destination



# Optional PyTorch/torchvision imports for MVTec-style data.
try:
    import torch
    from torch.utils.data import Dataset, DataLoader
    from torchvision import transforms
    from PIL import Image
    TORCH_AVAILABLE = True
except Exception:
    TORCH_AVAILABLE = False


if TORCH_AVAILABLE:
    class MVTecADDataset(Dataset):
        """MVTec AD loader: train uses only good; test uses good and defect categories."""
        def __init__(self, root: str | Path, category: str, split: str = "train", image_size: int = 256):
            self.root = Path(root) / category
            self.split = split
            if split not in {"train", "test"}:
                raise ValueError("split must be 'train' or 'test'")
            if not self.root.exists():
                raise FileNotFoundError(self.root)
            self.transform = transforms.Compose([
                transforms.Resize((image_size, image_size)),
                transforms.ToTensor(),
                transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ])
            self.mask_transform = transforms.Compose([
                transforms.Resize((image_size, image_size), interpolation=transforms.InterpolationMode.NEAREST),
                transforms.ToTensor(),
            ])
            self.samples = []
            self._collect_samples()

        def _find_mask(self, defect_name: str, image_path: Path) -> Optional[Path]:
            mask_dir = self.root / "ground_truth" / defect_name
            if not mask_dir.exists():
                return None
            candidates = [mask_dir / f"{image_path.stem}_mask.png", mask_dir / image_path.name]
            candidates += [mask_dir / f"{image_path.stem}_mask{ext}" for ext in IMAGE_EXTENSIONS]
            candidates += [mask_dir / f"{image_path.stem}{ext}" for ext in IMAGE_EXTENSIONS]
            return next((p for p in candidates if p.exists()), None)

        def _collect_samples(self):
            if self.split == "train":
                good_dir = self.root / "train" / "good"
                if not good_dir.exists():
                    raise FileNotFoundError(f"Expected MVTec training normals: {good_dir}")
                for p in sorted(good_dir.iterdir()):
                    if p.is_file() and is_image(p):
                        self.samples.append((p, None, 0, "good"))
            else:
                test_dir = self.root / "test"
                if not test_dir.exists():
                    raise FileNotFoundError(test_dir)
                for defect_dir in sorted(test_dir.iterdir()):
                    if not defect_dir.is_dir():
                        continue
                    defect_name = defect_dir.name
                    label = 0 if defect_name.lower() == "good" else 1
                    for p in sorted(defect_dir.iterdir()):
                        if p.is_file() and is_image(p):
                            mask = self._find_mask(defect_name, p) if label else None
                            self.samples.append((p, mask, label, defect_name))

        def __len__(self):
            return len(self.samples)

        def __getitem__(self, index):
            image_path, mask_path, label, defect_name = self.samples[index]
            image = Image.open(image_path).convert("RGB")
            image_tensor = self.transform(image)
            if mask_path is not None:
                mask_image = Image.open(mask_path).convert("L")
                mask_tensor = (self.mask_transform(mask_image) > 0).float()
            else:
                mask_tensor = torch.zeros((1, image_tensor.shape[-2], image_tensor.shape[-1]), dtype=torch.float32)
            return {
                "image": image_tensor,
                "mask": mask_tensor,
                "label": int(label),
                "defect_type": defect_name,
                "image_path": str(image_path),
            }

    def make_anomaly_dataloaders(root, category, batch_size=8, image_size=256, num_workers=0):
        train_ds = MVTecADDataset(root, category, "train", image_size)
        test_ds = MVTecADDataset(root, category, "test", image_size)
        return (
            DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers),
            DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers),
        )
else:
    MVTecADDataset = None
    make_anomaly_dataloaders = None




def demo_anomaly_dataset(root: Path, category: str, image_size: int, batch_size: int) -> None:
    if not TORCH_AVAILABLE:
        raise RuntimeError("PyTorch/torchvision are required for the MVTec demo loader.")
    train_loader, test_loader = make_anomaly_dataloaders(root, category, batch_size, image_size, num_workers=0)
    print(f"MVTec category={category} | train batches={len(train_loader)} | test batches={len(test_loader)}")
    if len(train_loader):
        batch=next(iter(train_loader))
        print("Train batch:", batch["image"].shape, "mask:", batch["mask"].shape)
    if len(test_loader):
        batch=next(iter(test_loader))
        print("Test batch:", batch["image"].shape, "labels:", batch["label"], "types:", batch["defect_type"])

def main():
    parser = argparse.ArgumentParser(description="Manufacturing defect dataset manager")
    sub = parser.add_subparsers(dest="command")
    sub.add_parser("list", help="List registered datasets")
    sub.add_parser("manual", help="Show official/manual dataset links")
    d = sub.add_parser("download", help="Download one or more datasets")
    d.add_argument("names", nargs="+")
    d.add_argument("--root", default="data/raw")
    d.add_argument("--force", action="store_true")
    d.add_argument("--yes", action="store_true", help="Required for large dataset ADNet")
    s = sub.add_parser("summarize", help="Summarize images under a folder")
    s.add_argument("--source", required=True)
    o = sub.add_parser("organize-classification", help="Create train/val/test class folders")
    o.add_argument("--source", required=True)
    o.add_argument("--dest", required=True)
    o.add_argument("--seed", type=int, default=42)
    o.add_argument("--train-ratio", type=float, default=.70)
    o.add_argument("--val-ratio", type=float, default=.15)
    anomaly_parser = sub.add_parser("demo-anomaly", help="Smoke-test an MVTec-style folder")
    anomaly_parser.add_argument("--root", default="data/raw/mvtec")
    anomaly_parser.add_argument("--category", default="hazelnut")
    anomaly_parser.add_argument("--image-size", type=int, default=256)
    anomaly_parser.add_argument("--batch-size", type=int, default=4)
    args = parser.parse_args()
    if args.command == "list":
        list_datasets()
    elif args.command == "manual":
        print_manual_datasets()
    elif args.command == "download":
        for name in args.names:
            if name in HUGE_DATASETS and not args.yes:
                print(f"Skipping {name}; add --yes to confirm large download.")
            else:
                download_dataset(name, Path(args.root), args.force)
    elif args.command == "summarize":
        summarize_folder(Path(args.source))
    elif args.command == "organize-classification":
        organize_classification_dataset(Path(args.source), Path(args.dest), args.seed, args.train_ratio, args.val_ratio)
    elif args.command == "demo-anomaly":
        demo_anomaly_dataset(Path(args.root), args.category, args.image_size, args.batch_size)
    else:
        parser.print_help()


if __name__ == "__main__":
    main()


## 4. Code module #2 — ML utilities

This module handles data adapters and validation. It supports YOLO-format datasets, Pascal VOC XML conversion, classification folder splits, box metrics/confidence sweeps, synthetic brightness/blur/noise checks, and tiled inference.

In [ ]:
%%writefile /content/industry4_qc/industry4_ml_utils.py
"""Reusable helpers for the Industry 4.0 quality-inspection Colab notebook."""
from __future__ import annotations
import math, random, shutil
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np
import cv2
import yaml
from PIL import Image

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
SPLIT_ALIASES = {'train':'train', 'training':'train', 'val':'val', 'valid':'val', 'validation':'val', 'test':'test', 'testing':'test'}


def list_images(root):
    root = Path(root)
    return sorted(p for p in root.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS) if root.exists() else []


def class_root_candidates(root):
    """Find ImageFolder-like roots: 2+ immediate folders each containing images."""
    root = Path(root)
    if not root.exists():
        return []
    directories = [root] + [p for p in root.rglob('*') if p.is_dir()]
    candidates = []
    for parent in directories:
        try:
            subdirs = [d for d in parent.iterdir() if d.is_dir()]
        except OSError:
            continue
        classes = []
        for d in subdirs:
            images = [p for p in d.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS]
            if images:
                classes.append(d)
        if len(classes) >= 2:
            count = sum(sum(1 for p in d.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS) for d in classes)
            split = None
            for ancestor in [parent] + list(parent.parents):
                if ancestor.name.lower() in SPLIT_ALIASES:
                    split = SPLIT_ALIASES[ancestor.name.lower()]
                    break
            candidates.append({'root': parent, 'classes': classes, 'count': count, 'split': split})
    return candidates


def _image_files_by_class(root):
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(root)
    mapping = {}
    for class_dir in sorted(p for p in root.iterdir() if p.is_dir()):
        images = sorted(p for p in class_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
        if images:
            mapping[class_dir.name] = images
    if len(mapping) < 2:
        raise ValueError(f'Expected >=2 image class folders under {root}')
    return mapping


def _copy_class_mapping(mapping, dest_split, class_name):
    target = Path(dest_split) / class_name
    target.mkdir(parents=True, exist_ok=True)
    for idx, source in enumerate(mapping):
        shutil.copy2(source, target / f'{idx:06d}_{source.name}')


def _split_files(files, seed, train_ratio=.70, val_ratio=.15):
    files = list(files)
    random.Random(seed).shuffle(files)
    n = len(files)
    if n < 3:
        raise ValueError(f'Each classification class requires at least 3 images; got {n}')
    n_train = max(1, int(round(n * train_ratio)))
    n_val = max(1, int(round(n * val_ratio)))
    n_test = n - n_train - n_val
    if n_test < 1:
        n_test = 1
        if n_train > 1:
            n_train -= 1
        else:
            n_val -= 1
    return {'train': files[:n_train], 'val': files[n_train:n_train+n_val], 'test': files[n_train+n_val:]}


def prepare_classification_splits(raw_root, processed_root, seed=42, train_ratio=.70, val_ratio=.15):
    """Discover class folders and prepare train/val/test. Existing named splits are preserved.


    When only train+validation exist, training is split into train/val and the
    original validation set becomes test to avoid using it twice.
    """
    raw_root, processed_root = Path(raw_root), Path(processed_root)
    candidates = class_root_candidates(raw_root)
    if not candidates:
        raise ValueError(f'No class-folder images found beneath {raw_root}. Expected folders like class_name/image.jpg')
    split_candidates = [c for c in candidates if c['split']]
    best_per_split = {}
    for candidate in split_candidates:
        split = candidate['split']
        if split not in best_per_split or candidate['count'] > best_per_split[split]['count']:
            best_per_split[split] = candidate
    classes = None
    split_sources = {}
    if 'train' in best_per_split and ('val' in best_per_split or 'test' in best_per_split):
        train_map = _image_files_by_class(best_per_split['train']['root'])
        classes = sorted(train_map)
        for split, candidate in best_per_split.items():
            class_map = _image_files_by_class(candidate['root'])
            if sorted(class_map) != classes:
                raise ValueError(f'Class folders differ between splits ({split}); inspect dataset manually.')
            split_sources[split] = class_map
    else:
        flat_candidate = max((c for c in candidates if not c['split']), key=lambda c:c['count'], default=None)
        if flat_candidate is None:
            # Fall back to largest class-root candidate, but never mix split roots.
            flat_candidate = max(candidates, key=lambda c:c['count'])
        classes = sorted(_image_files_by_class(flat_candidate['root']))
        split_sources['flat'] = _image_files_by_class(flat_candidate['root'])

    if processed_root.exists():
        shutil.rmtree(processed_root)
    processed_root.mkdir(parents=True, exist_ok=True)

    if 'flat' in split_sources:
        for class_name, files in split_sources['flat'].items():
            splits = _split_files(files, seed, train_ratio, val_ratio)
            for split_name, split_files in splits.items():
                _copy_class_mapping(split_files, processed_root / split_name, class_name)
    else:
        # Case: a fully supplied train/val/test structure.
        if {'train','val','test'}.issubset(split_sources):
            for split_name, class_map in split_sources.items():
                for class_name, files in class_map.items():
                    _copy_class_mapping(files, processed_root / split_name, class_name)
        # Case: train+val, but no independent test. Carve val from train and reserve
        # original val for test. This is a pragmatic benchmark split, not factory batch split.
        elif 'train' in split_sources and 'val' in split_sources and 'test' not in split_sources:
            for class_name, files in split_sources['train'].items():
                train_val = _split_files(files, seed, train_ratio=.85, val_ratio=.15)
                for split_name in ('train','val'):
                    _copy_class_mapping(train_val[split_name], processed_root / split_name, class_name)
                _copy_class_mapping(split_sources['val'][class_name], processed_root / 'test', class_name)
        # Case: train+test. Split train into train/val and retain official test.
        elif 'train' in split_sources and 'test' in split_sources:
            for class_name, files in split_sources['train'].items():
                train_val = _split_files(files, seed, train_ratio=.85, val_ratio=.15)
                for split_name in ('train','val'):
                    _copy_class_mapping(train_val[split_name], processed_root / split_name, class_name)
                _copy_class_mapping(split_sources['test'][class_name], processed_root / 'test', class_name)
        else:
            # Only one split root found; use it as a flat class-folder dataset.
            largest = max(candidates, key=lambda c:c['count'])
            class_map = _image_files_by_class(largest['root'])
            for class_name, files in class_map.items():
                splits = _split_files(files, seed, train_ratio, val_ratio)
                for split_name, split_files in splits.items():
                    _copy_class_mapping(split_files, processed_root / split_name, class_name)
    # Final structural check for all splits.
    for split in ('train','val','test'):
        split_dir = processed_root / split
        if not split_dir.exists() or not any(p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS for p in split_dir.rglob('*')):
            raise RuntimeError(f'Classification split {split!r} is empty at {split_dir}')
    print('Classification data ready:', processed_root)
    return processed_root


def find_yolo_yaml(root):
    root = Path(root)
    configs = sorted(list(root.rglob('data.yaml')) + list(root.rglob('dataset.yaml')) + list(root.rglob('data.yml')))
    for cfg in configs:
        try:
            parsed = yaml.safe_load(cfg.read_text(encoding='utf-8')) or {}
        except Exception:
            continue
        if isinstance(parsed, dict) and 'train' in parsed and 'val' in parsed and 'names' in parsed:
            return cfg
    return None


def _resolve_dir(value, base, yaml_parent):
    if isinstance(value, list):
        raise ValueError('This notebook adapter expects a single image directory per split; YAML lists are not supported yet.')
    path = Path(str(value))
    candidates = []
    if path.is_absolute():
        candidates.append(path)
    else:
        candidates.extend([Path(base) / path, Path(yaml_parent) / path, Path(yaml_parent) / path.name])
    for candidate in candidates:
        if candidate.exists() and candidate.is_dir():
            return candidate.resolve()
    raise FileNotFoundError(f'Cannot resolve split folder {value!r}. Checked: {candidates}')


def derive_label_dir(image_dir):
    image_dir = Path(image_dir)
    parts = list(image_dir.parts)
    indices = [i for i, part in enumerate(parts) if part.lower() == 'images']
    if indices:
        parts[indices[-1]] = 'labels'
        return Path(*parts)
    # Common split layout: dataset/<split>/images and dataset/<split>/labels.
    return image_dir.parent / 'labels'


def validate_yolo_label_file(path, class_count):
    path = Path(path)
    if not path.exists():
        return 0
    boxes = 0
    for line_no, line in enumerate(path.read_text(encoding='utf-8', errors='ignore').splitlines(), start=1):
        if not line.strip():
            continue
        parts = line.split()
        if len(parts) != 5:
            raise ValueError(f'{path}:{line_no} expected 5 fields, got {len(parts)}')
        try:
            class_id_float = float(parts[0])
            class_id = int(class_id_float)
            if class_id_float != class_id:
                raise ValueError('class id must be an integer')
            coords = [float(v) for v in parts[1:]]
        except ValueError as exc:
            raise ValueError(f'{path}:{line_no} contains non-numeric label data') from exc
        if class_id < 0 or class_id >= class_count:
            raise ValueError(f'{path}:{line_no} class id {class_id} outside 0..{class_count-1}')
        if not all(math.isfinite(v) and 0.0 <= v <= 1.0 for v in coords):
            raise ValueError(f'{path}:{line_no} coordinates must be finite and normalized to [0,1]')
        cx, cy, width, height = coords
        if width <= 0 or height <= 0:
            raise ValueError(f'{path}:{line_no} box width/height must be positive')
        if cx-width/2 < -1e-6 or cx+width/2 > 1.000001 or cy-height/2 < -1e-6 or cy+height/2 > 1.000001:
            raise ValueError(f'{path}:{line_no} box extends outside normalized image bounds')
        boxes += 1
    return boxes


def normalize_yolo_yaml(yaml_path, output_yaml, root_override=None):
    """Normalize YOLO paths to absolute paths and validate train/val/test labels."""
    yaml_path, output_yaml = Path(yaml_path), Path(output_yaml)
    cfg = yaml.safe_load(yaml_path.read_text(encoding='utf-8'))
    if not isinstance(cfg, dict):
        raise ValueError(f'Invalid YAML mapping: {yaml_path}')
    raw_names = cfg.get('names')
    if isinstance(raw_names, dict):
        try:
            keys = sorted(raw_names, key=lambda x:int(x))
        except (TypeError, ValueError):
            keys = sorted(raw_names, key=lambda x:str(x))
        names = [str(raw_names[k]) for k in keys]
    elif isinstance(raw_names, list):
        names = [str(x) for x in raw_names]
    else:
        raise ValueError(f'{yaml_path} must contain class names in names: list or mapping')
    if not names or len(set(names)) != len(names):
        raise ValueError('YOLO class names must be non-empty and unique')
    base = Path(root_override) if root_override else Path(cfg.get('path', yaml_path.parent))
    if not base.is_absolute():
        base = (yaml_path.parent / base).resolve()
    if not base.exists():
        base = yaml_path.parent.resolve()
    train_dir = _resolve_dir(cfg['train'], base, yaml_path.parent)
    val_dir = _resolve_dir(cfg['val'], base, yaml_path.parent)
    test_dir = None
    if cfg.get('test'):
        try:
            candidate_test_dir = _resolve_dir(cfg['test'], base, yaml_path.parent)
            if list_images(candidate_test_dir):
                test_dir = candidate_test_dir
            else:
                print('Warning: test split directory is empty; final test evaluation will be skipped.')
        except FileNotFoundError:
            print('Warning: test split path not found; final test evaluation will be skipped.')
    split_dirs = {'train':train_dir,'val':val_dir}
    if test_dir is not None:
        split_dirs['test'] = test_dir
    summary = {}
    for split, image_dir in split_dirs.items():
        images = list_images(image_dir)
        if not images:
            raise ValueError(f'{split} split has no images: {image_dir}')
        label_dir = derive_label_dir(image_dir)
        if split in ('train','val') and not label_dir.exists():
            raise FileNotFoundError(f'{split} label folder missing: {label_dir}')
        label_files = list(label_dir.rglob('*.txt')) if label_dir.exists() else []
        total_boxes = 0
        for label_file in label_files:
            total_boxes += validate_yolo_label_file(label_file, len(names))
        if split in ('train','val') and not label_files:
            raise ValueError(f'{split} has no label files under {label_dir}')
        if split in ('train','val') and total_boxes == 0:
            raise ValueError(f'{split} split contains no valid boxes')
        if split=='test' and not label_files:
            print(f'Warning: test labels are absent/empty at {label_dir}; test metrics may be invalid if defects exist.')
        summary[split] = {'images':len(images),'label_files':len(label_files),'boxes':total_boxes,'image_dir':str(image_dir),'label_dir':str(label_dir)}
    normalized = {'path':str(base.resolve()),'train':str(train_dir),'val':str(val_dir),'names':names}
    if test_dir is not None:
        normalized['test'] = str(test_dir)
    output_yaml.parent.mkdir(parents=True, exist_ok=True)
    output_yaml.write_text(yaml.safe_dump(normalized,sort_keys=False),encoding='utf-8')
    print('Normalized YOLO data YAML:',output_yaml)
    return output_yaml, names, summary


def convert_voc_xml_to_yolo(raw_root, output_root, seed=42, train_ratio=.70, val_ratio=.15):
    """Convert paired Pascal VOC XML/images into YOLO folders with a reproducible split."""
    raw_root, output_root = Path(raw_root), Path(output_root)
    xml_files = sorted(raw_root.rglob('*.xml'))
    image_files = list_images(raw_root)
    if not xml_files or not image_files:
        return None
    by_stem = defaultdict(list)
    for image in image_files:
        by_stem[image.stem.lower()].append(image)
    samples = []
    class_set = set()
    for xml in xml_files:
        matches = by_stem.get(xml.stem.lower(), [])
        if len(matches) != 1:
            continue
        try:
            import xml.etree.ElementTree as ET
            root = ET.parse(xml).getroot()
        except Exception:
            continue
        image = matches[0]
        with Image.open(image) as im:
            width, height = im.size
        boxes = []
        for obj in root.findall('object'):
            class_name = (obj.findtext('name') or '').strip()
            bnd = obj.find('bndbox')
            if not class_name or bnd is None:
                continue
            try:
                x1 = float(bnd.findtext('xmin')); y1 = float(bnd.findtext('ymin'))
                x2 = float(bnd.findtext('xmax')); y2 = float(bnd.findtext('ymax'))
            except (TypeError,ValueError):
                continue
            x1=max(0.0,min(x1,width-1)); y1=max(0.0,min(y1,height-1))
            x2=max(0.0,min(x2,width)); y2=max(0.0,min(y2,height))
            if x2<=x1 or y2<=y1:
                continue
            boxes.append((class_name,x1,y1,x2,y2))
            class_set.add(class_name)
        if boxes:
            samples.append({'image':image,'boxes':boxes,'width':width,'height':height})
    if len(samples)<30 or not class_set:
        return None
    if output_root.exists():
        shutil.rmtree(output_root)
    names=sorted(class_set)
    class_to_id={name:i for i,name in enumerate(names)}
    # Use dominant class to stratify when enough examples exist; otherwise fall back to seeded random.
    strata=[]
    for sample in samples:
        counts=Counter(b[0] for b in sample['boxes'])
        strata.append(counts.most_common(1)[0][0])
    try:
        from sklearn.model_selection import train_test_split
        indices=list(range(len(samples)))
        train_idx,remainder_idx=train_test_split(indices,test_size=(1-train_ratio),random_state=seed,stratify=strata)
        remainder_strata=[strata[i] for i in remainder_idx]
        test_fraction=val_ratio/(1-train_ratio)
        val_idx,test_idx=train_test_split(remainder_idx,test_size=test_fraction,random_state=seed,stratify=remainder_strata)
        train=[samples[i] for i in train_idx]; val=[samples[i] for i in val_idx]; test=[samples[i] for i in test_idx]
    except Exception:
        rng=random.Random(seed); shuffled=list(samples); rng.shuffle(shuffled)
        n_train=max(1,int(len(shuffled)*train_ratio)); n_val=max(1,int(len(shuffled)*val_ratio))
        train=shuffled[:n_train]; val=shuffled[n_train:n_train+n_val]; test=shuffled[n_train+n_val:]
        if not test: test=val[-1:]; val=val[:-1]
    for split,rows in [('train',train),('val',val),('test',test)]:
        (output_root/'images'/split).mkdir(parents=True,exist_ok=True)
        (output_root/'labels'/split).mkdir(parents=True,exist_ok=True)
        for idx,sample in enumerate(rows):
            image=sample['image']; name=f'{image.stem}_{idx:05d}'
            shutil.copy2(image,output_root/'images'/split/f'{name}{image.suffix.lower()}')
            lines=[]
            for class_name,x1,y1,x2,y2 in sample['boxes']:
                lines.append(f'{class_to_id[class_name]} {(x1+x2)/(2*sample["width"]):.6f} {(y1+y2)/(2*sample["height"]):.6f} {(x2-x1)/sample["width"]:.6f} {(y2-y1)/sample["height"]:.6f}')
            (output_root/'labels'/split/f'{name}.txt').write_text('\n'.join(lines),encoding='utf-8')
    yaml_path=output_root/'data.yaml'
    cfg={'path':str(output_root.resolve()),'train':'images/train','val':'images/val','test':'images/test','names':names}
    yaml_path.write_text(yaml.safe_dump(cfg,sort_keys=False),encoding='utf-8')
    return yaml_path


def yolo_label_path(image_path, image_dir, label_dir):
    image_path=Path(image_path); image_dir=Path(image_dir); label_dir=Path(label_dir)
    rel=image_path.relative_to(image_dir)
    return label_dir/rel.with_suffix('.txt')


def read_yolo_gt(image_path, image_dir, label_dir):
    image_path=Path(image_path)
    image=cv2.imread(str(image_path))
    if image is None:
        raise ValueError(f'Cannot read image: {image_path}')
    height,width=image.shape[:2]
    label_path=yolo_label_path(image_path,image_dir,label_dir)
    boxes=[]
    if label_path.exists():
        for line in label_path.read_text(encoding='utf-8',errors='ignore').splitlines():
            if not line.strip(): continue
            cls,cx,cy,bw,bh=map(float,line.split())
            boxes.append((int(cls),[max(0,(cx-bw/2)*width),max(0,(cy-bh/2)*height),min(width,(cx+bw/2)*width),min(height,(cy+bh/2)*height)]))
    return boxes,(width,height)


def box_iou_xyxy(a,b):
    x1=max(float(a[0]),float(b[0])); y1=max(float(a[1]),float(b[1]))
    x2=min(float(a[2]),float(b[2])); y2=min(float(a[3]),float(b[3]))
    inter=max(0,x2-x1)*max(0,y2-y1)
    area_a=max(0,float(a[2])-float(a[0]))*max(0,float(a[3])-float(a[1]))
    area_b=max(0,float(b[2])-float(b[0]))*max(0,float(b[3])-float(b[1]))
    return inter/max(area_a+area_b-inter,1e-9)


def cache_detection_predictions(model, image_paths, imgsz=640, device='cpu', low_conf=.01):
    """Cache predictions once at a low confidence; later confidence sweeps are cheap."""
    cache=[]
    for image_path in image_paths:
        result=model.predict(source=str(image_path),imgsz=imgsz,conf=low_conf,device=device,verbose=False)[0]
        pred=[]
        if result.boxes is not None and len(result.boxes):
            for box in result.boxes:
                pred.append((int(box.cls.item()),box.xyxy[0].detach().cpu().numpy().astype(float),float(box.conf.item())))
        cache.append({'image_path':Path(image_path),'predictions':pred})
    return cache


def detection_metrics_from_cache(cache, image_dir, label_dir, confidence=.25, iou_threshold=.5):
    totals={'tp':0,'fp':0,'fn':0,'images':0,'gt_boxes':0}
    for item in cache:
        image_path=item['image_path']
        gt,_=read_yolo_gt(image_path,image_dir,label_dir)
        preds=[p for p in item['predictions'] if p[2]>=confidence]
        preds=sorted(preds,key=lambda p:p[2],reverse=True)
        unmatched=set(range(len(gt)))
        tp=fp=0
        for cls,pbox,_score in preds:
            choices=[(idx,box_iou_xyxy(pbox,gbox)) for idx,(gcls,gbox) in enumerate(gt) if idx in unmatched and gcls==cls]
            best=max(choices,key=lambda x:x[1],default=None)
            if best is not None and best[1]>=iou_threshold:
                unmatched.remove(best[0]); tp+=1
            else:
                fp+=1
        fn=len(unmatched)
        totals['tp']+=tp; totals['fp']+=fp; totals['fn']+=fn; totals['images']+=1; totals['gt_boxes']+=len(gt)
    precision=totals['tp']/max(totals['tp']+totals['fp'],1)
    recall=totals['tp']/max(totals['tp']+totals['fn'],1)
    f1=2*precision*recall/max(precision+recall,1e-12)
    totals.update({'precision':precision,'recall':recall,'f1':f1,'confidence':confidence,'iou_threshold':iou_threshold})
    return totals


def confidence_sweep(cache, image_dir, label_dir, thresholds=None, target_recall=.95):
    if thresholds is None:
        thresholds=np.arange(.05,.51,.05).round(2).tolist()
    rows=[detection_metrics_from_cache(cache,image_dir,label_dir,float(t)) for t in thresholds]
    feasible=[r for r in rows if r['recall']>=target_recall]
    chosen=max(feasible,key=lambda r:r['precision']) if feasible else max(rows,key=lambda r:r['f1'])
    return rows,chosen


def perturb_image(image, kind, seed=42):
    image=np.asarray(image).copy()
    if kind=='original': return image
    if kind=='dark': return cv2.convertScaleAbs(image,alpha=.65,beta=0)
    if kind=='bright': return cv2.convertScaleAbs(image,alpha=1.15,beta=20)
    if kind=='contrast_low': return cv2.convertScaleAbs(image,alpha=.75,beta=35)
    if kind=='blur': return cv2.GaussianBlur(image,(5,5),1.4)
    if kind=='noise':
        rng=np.random.default_rng(seed)
        return np.clip(image.astype(np.float32)+rng.normal(0,12,image.shape),0,255).astype(np.uint8)
    raise KeyError(f'Unknown perturbation: {kind}')


def image_quality_stats(image):
    image=np.asarray(image)
    if image.ndim==2:
        gray=image
    else:
        gray=cv2.cvtColor(image,cv2.COLOR_BGR2GRAY)
    return {'brightness':float(gray.mean()),'contrast':float(gray.std()),'sharpness':float(cv2.Laplacian(gray,cv2.CV_64F).var())}


def tiled_predict(model, image_bgr, class_names=None, tile_size=640, overlap=.20, imgsz=640, conf=.25, device='cpu'):
    """Tile large images, offset boxes, and merge with class-wise greedy NMS."""
    if tile_size<=0 or not (0.0<=overlap<1.0):
        raise ValueError('tile_size must be positive and overlap must be in [0,1)')
    h,w=image_bgr.shape[:2]
    if h<=tile_size and w<=tile_size:
        results=model.predict(source=image_bgr,imgsz=imgsz,conf=conf,device=device,verbose=False)[0]
        detections=[]
        for b in results.boxes:
            cls=int(b.cls.item()); box=b.xyxy[0].cpu().numpy().astype(float)
            detections.append({'class_id':cls,'class_name':(class_names or model.names)[cls],'confidence':float(b.conf.item()),'bbox_xyxy':box.tolist()})
        return detections
    step=max(1,int(tile_size*(1-overlap)))
    def starts(length):
        if length<=tile_size: return [0]
        s=list(range(0,length-tile_size+1,step))
        if not s or s[-1]!=length-tile_size: s.append(length-tile_size)
        return s
    candidates=[]
    for y in starts(h):
        for x in starts(w):
            tile=image_bgr[y:min(y+tile_size,h),x:min(x+tile_size,w)]
            result=model.predict(source=tile,imgsz=imgsz,conf=conf,device=device,verbose=False)[0]
            for b in result.boxes:
                cls=int(b.cls.item()); box=b.xyxy[0].cpu().numpy().astype(float)
                box[[0,2]]+=x; box[[1,3]]+=y
                candidates.append({'class_id':cls,'class_name':(class_names or model.names)[cls],'confidence':float(b.conf.item()),'bbox_xyxy':box.tolist()})
    # Greedy, class-specific NMS to remove duplicates across overlapping tiles.
    kept=[]
    for cls_id in sorted(set(d['class_id'] for d in candidates)):
        group=sorted((d for d in candidates if d['class_id']==cls_id),key=lambda d:d['confidence'],reverse=True)
        while group:
            best=group.pop(0); kept.append(best)
            group=[d for d in group if box_iou_xyxy(best['bbox_xyxy'],d['bbox_xyxy'])<.50]
    return kept


## 5. Import the modules and inspect the registry

No data is downloaded by this cell.

In [ ]:
import importlib
import dataset_manager as dm
import industry4_ml_utils as ml
importlib.reload(dm); importlib.reload(ml)
dm.list_datasets()
print('\nManual dataset options:')
dm.print_manual_datasets()

## 6. Download or point to the dataset

`gc10_hf` is the recommended default because it is a defect-localization dataset with YOLO-format annotations. Kaggle downloads may require an API token; if so, upload `kaggle.json` into Colab and place it at `~/.kaggle/kaggle.json` with permissions restricted to the owner. Never commit your token.

In [ ]:
RAW_ROOT = ROOT / 'data' / 'raw'
if DATA_MODE == 'custom_yolo':
    raw = None
    if not CUSTOM_YOLO_YAML.is_file():
        raise FileNotFoundError(f'Custom data.yaml not found: {CUSTOM_YOLO_YAML}. Upload it or mount Google Drive.')
    print('Using custom dataset YAML:', CUSTOM_YOLO_YAML)
else:
    if DATA_MODE not in dm.DATASETS:
        raise ValueError(f'Unknown DATA_MODE={DATA_MODE!r}. Choices: {sorted(dm.DATASETS)}')
    raw = dm.download_dataset(DATA_MODE, root=RAW_ROOT)
    dm.summarize_folder(raw)

## 7. Detect the correct task and prepare the data

**Detection** needs actual boxes in YOLO text files or Pascal VOC XML. If a dataset contains only class-folder labels, it is routed to image classification. Segmentation-only Severstal is intentionally not converted into boxes.

In [ ]:
TASK = None
DATA_YAML = None
CLASSES = None
YOLO_SUMMARY = None
CLASSIFICATION_ROOT = None

if DATA_MODE == 'custom_yolo':
    TASK = 'detection'
    DATA_YAML, CLASSES, YOLO_SUMMARY = ml.normalize_yolo_yaml(
        CUSTOM_YOLO_YAML, ROOT/'custom_resolved.yaml')
elif DATA_MODE == 'gc10_hf':
    yaml_path = ml.find_yolo_yaml(raw)
    if yaml_path is None:
        raise FileNotFoundError(f'GC10-HF download is missing data.yaml: {raw}. Check the dataset repository structure.')
    TASK = 'detection'
    DATA_YAML, CLASSES, YOLO_SUMMARY = ml.normalize_yolo_yaml(
        yaml_path, ROOT/'gc10_hf_resolved.yaml')
elif DATA_MODE == 'severstal_hf':
    raise NotImplementedError('Severstal is segmentation data in this registry. This notebook does not convert masks into fake bounding boxes; use a segmentation model/adapter.')
elif DATA_MODE in {'mvtec_hf', 'adnet'}:
    raise NotImplementedError('These registered sources are anomaly datasets and are not routed through the supervised detector/classifier branch. Use the optional anomaly section with an inspected folder structure; ADNet is large.')
else:
    # Kaggle sources: use a YOLO YAML if available, otherwise real VOC XML,
    # otherwise class folders (classification). No synthetic boxes are created.
    yaml_path = ml.find_yolo_yaml(raw)
    if yaml_path is not None:
        TASK = 'detection'
        DATA_YAML, CLASSES, YOLO_SUMMARY = ml.normalize_yolo_yaml(
            yaml_path, ROOT/f'{DATA_MODE}_resolved.yaml')
    else:
        voc_yaml = ml.convert_voc_xml_to_yolo(raw, ROOT/f'{DATA_MODE}_voc_yolo', seed=SEED)
        if voc_yaml is not None:
            TASK = 'detection'
            DATA_YAML, CLASSES, YOLO_SUMMARY = ml.normalize_yolo_yaml(
                voc_yaml, ROOT/f'{DATA_MODE}_voc_resolved.yaml')
        else:
            TASK = 'classification'
            CLASSIFICATION_ROOT = ROOT/'data'/'processed'/'classification'/DATA_MODE

print('Selected task:', TASK)
if TASK == 'detection':
    print('Classes:', CLASSES)
    display(pd.DataFrame(YOLO_SUMMARY).T)

## 8. Smoke tests for data preparation code

These tests use small synthetic files only. They do **not** train a model and do not count as accuracy results. They verify class-folder splitting, YAML path normalization, box-label validation, and VOC conversion before touching the real dataset.

In [ ]:
from tempfile import TemporaryDirectory
from xml.etree.ElementTree import Element, SubElement, ElementTree

with TemporaryDirectory(prefix='industry4_smoke_') as tmp:
    tmp = Path(tmp)
    # Classification folders -> non-empty train/val/test
    flat = tmp/'flat_classes'
    for cname in ['defect_a','defect_b']:
        (flat/cname).mkdir(parents=True)
        for i in range(12):
            Image.fromarray(np.full((24,24,3),i*10,dtype=np.uint8)).save(flat/cname/f'{i}.png')
    cls_out = ml.prepare_classification_splits(flat,tmp/'class_split',seed=SEED)
    assert all(ml.list_images(cls_out/s) for s in ['train','val','test'])
    # YOLO label validation and synthetic directory/YAML
    ds = tmp/'yolo_ds'
    for split in ['train','val','test']:
        (ds/'images'/split).mkdir(parents=True); (ds/'labels'/split).mkdir(parents=True)
        Image.fromarray(np.zeros((64,64,3),dtype=np.uint8)).save(ds/'images'/split/'sample.png')
        (ds/'labels'/split/'sample.txt').write_text('0 0.5 0.5 0.2 0.3\n')
    yaml_path = ds/'data.yaml'
    yaml_path.write_text(yaml.safe_dump({'path':str(ds),'train':'images/train','val':'images/val','test':'images/test','names':['scratch']}))
    resolved, names, summary = ml.normalize_yolo_yaml(yaml_path,tmp/'resolved.yaml')
    assert names == ['scratch'] and summary['test']['boxes'] == 1
    assert ml.validate_yolo_label_file(ds/'labels'/'train'/'sample.txt',1) == 1
    # VOC XML adapter with enough samples to exercise split and export
    voc = tmp/'voc'; voc.mkdir()
    for i in range(36):
        im_path=voc/f'img_{i:03d}.jpg'
        Image.fromarray(np.zeros((80,80,3),dtype=np.uint8)).save(im_path)
        root_xml=Element('annotation'); SubElement(root_xml,'filename').text=im_path.name
        size=SubElement(root_xml,'size'); SubElement(size,'width').text='80'; SubElement(size,'height').text='80'; SubElement(size,'depth').text='3'
        obj=SubElement(root_xml,'object'); SubElement(obj,'name').text='scratch' if i%2 else 'dent'
        box=SubElement(obj,'bndbox'); SubElement(box,'xmin').text='10'; SubElement(box,'ymin').text='10'; SubElement(box,'xmax').text='40'; SubElement(box,'ymax').text='45'
        ElementTree(root_xml).write(voc/f'img_{i:03d}.xml')
    converted = ml.convert_voc_xml_to_yolo(voc,tmp/'voc_yolo',seed=SEED)
    assert converted is not None
    print('Smoke tests passed: classification split, YOLO YAML/labels, and VOC conversion.')

## 9. Dataset audit — before training

The audit displays counts and class distribution. Manually review the sampled images and annotations. Incorrect labels and leakage are often bigger accuracy problems than the model architecture.

In [ ]:
if TASK == 'detection':
    audit_rows=[]
    cross_split_hashes={}
    corrupt_images=[]
    for split,info in YOLO_SUMMARY.items():
        split_paths=ml.list_images(info['image_dir'])
        audit_rows.append({'split':split,'images':info['images'],'label_files':info['label_files'],'boxes':info['boxes']})
        for image_path in split_paths:
            try:
                with Image.open(image_path) as im: im.verify()
            except Exception:
                corrupt_images.append(str(image_path))
            digest=hashlib.sha256(image_path.read_bytes()).hexdigest()
            cross_split_hashes.setdefault(digest,set()).add(split)
    if corrupt_images:
        raise RuntimeError(f'Found unreadable images. Remove/fix before training: {corrupt_images[:10]}')
    duplicate_cross_split=[splits for splits in cross_split_hashes.values() if len(splits)>1]
    print('Exact duplicate image content across splits:',len(duplicate_cross_split))
    if duplicate_cross_split: print('WARNING: duplicate images across splits can inflate test performance; remove duplicates or regroup before final evaluation.')
    display(pd.DataFrame(audit_rows))
    print('Dataset YAML:'); print(DATA_YAML.read_text())
    train_image_dir=Path(YOLO_SUMMARY['train']['image_dir'])
    train_label_dir=Path(YOLO_SUMMARY['train']['label_dir'])
    sample_images=ml.list_images(train_image_dir)[:6]
    fig,axes=plt.subplots(2,3,figsize=(14,7))
    for ax,img_path in zip(axes.ravel(),sample_images):
        bgr=cv2.imread(str(img_path)); h,w=bgr.shape[:2]
        lab=ml.yolo_label_path(img_path,train_image_dir,train_label_dir)
        if lab.exists():
            for line in lab.read_text().splitlines():
                if not line.strip(): continue
                cls,cx,cy,bw,bh=map(float,line.split())
                x1=int((cx-bw/2)*w); y1=int((cy-bh/2)*h); x2=int((cx+bw/2)*w); y2=int((cy+bh/2)*h)
                cv2.rectangle(bgr,(x1,y1),(x2,y2),(0,255,0),2)
                cv2.putText(bgr,CLASSES[int(cls)],(x1,max(15,y1-4)),cv2.FONT_HERSHEY_SIMPLEX,.45,(0,255,0),1)
        ax.imshow(cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB)); ax.set_title(img_path.name); ax.axis('off')
    plt.tight_layout(); plt.show()
else:
    print('Task is image classification. Folder class names are inspected in the classification preparation cell.')

## 10. Train the known-defect detector (YOLO11)

A small YOLO model is the baseline for edge deployment. Try `yolo11s.pt` or a larger image size only after benchmarking recall, false alarms and latency. For tiny cracks, image tiling during inference may help.

In [ ]:
detector = None
BEST_WEIGHTS = None
train_results = None
if TASK == 'detection' and RUN_TRAINING:
    from ultralytics import YOLO
    detector = YOLO(MODEL_NAME)
    train_results = detector.train(
        data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH_SIZE,
        device=DEVICE, project=str(ROOT/'runs'), name=f'{DATA_MODE}_yolo11n',
        exist_ok=True, patience=15, seed=SEED, workers=WORKERS, plots=True,
        degrees=2.0, translate=0.02, scale=0.12, fliplr=0.5, flipud=0.0,
        hsv_h=0.0, hsv_s=0.05, hsv_v=0.15, mosaic=0.10, mixup=0.0,
        close_mosaic=10, verbose=False)
    BEST_WEIGHTS=Path(detector.trainer.best)
    if not BEST_WEIGHTS.is_file():
        raise FileNotFoundError(f'YOLO best checkpoint missing: {BEST_WEIGHTS}')
    detector=YOLO(str(BEST_WEIGHTS))
    print('Best checkpoint:',BEST_WEIGHTS)
elif TASK == 'detection':
    print('Training is disabled. Set RUN_TRAINING=True to train the detector.')
else:
    print('Detector training skipped: classification task selected.')

## 11. Official detector evaluation

The held-out test split is evaluated **after training**. Do not use it to tune augmentation, thresholds or architecture.

In [ ]:
OFFICIAL_TEST_METRICS = None
TEST_IMAGE_DIR = None
TEST_LABEL_DIR = None
if TASK == 'detection' and detector is not None:
    if 'test' in YOLO_SUMMARY:
        OFFICIAL_TEST_METRICS=detector.val(data=str(DATA_YAML),split='test',imgsz=IMGSZ,
                                         batch=BATCH_SIZE,device=DEVICE,plots=True,verbose=False)
        print({'mAP50':float(OFFICIAL_TEST_METRICS.box.map50),
              'mAP50-95':float(OFFICIAL_TEST_METRICS.box.map),
              'precision':float(OFFICIAL_TEST_METRICS.box.mp),
              'recall':float(OFFICIAL_TEST_METRICS.box.mr)})
        TEST_IMAGE_DIR=Path(YOLO_SUMMARY['test']['image_dir'])
        TEST_LABEL_DIR=Path(YOLO_SUMMARY['test']['label_dir'])
    else:
        print('No independent test split found; only validation results are available. Do not label validation as test.')
else:
    print('Detector evaluation skipped.')

## 12. Calibrate the confidence threshold using validation only

This is a custom box-matching metric at IoU 0.50. It sweeps prediction confidence on **validation images only**, chooses the highest precision threshold that achieves the requested recall (or highest F1 if the target recall cannot be reached), then evaluates that chosen threshold on test.

In [ ]:
CONF_THRESHOLD = 0.25
VAL_CACHE = None
TEST_CACHE = None
CONF_SWEEP_TABLE = None
VAL_IMAGE_DIR = None
VAL_LABEL_DIR = None
if TASK == 'detection' and detector is not None:
    VAL_IMAGE_DIR=Path(YOLO_SUMMARY['val']['image_dir'])
    VAL_LABEL_DIR=Path(YOLO_SUMMARY['val']['label_dir'])
    val_images=ml.list_images(VAL_IMAGE_DIR)
    if not val_images: raise RuntimeError('Validation image folder is empty.')
    VAL_CACHE=ml.cache_detection_predictions(detector,val_images,imgsz=IMGSZ,device=DEVICE,low_conf=.01)
    sweep,chosen=ml.confidence_sweep(VAL_CACHE,VAL_IMAGE_DIR,VAL_LABEL_DIR,
                                   thresholds=np.arange(.05,.51,.05),target_recall=.95)
    CONF_SWEEP_TABLE=pd.DataFrame(sweep)
    display(CONF_SWEEP_TABLE[['confidence','precision','recall','f1','tp','fp','fn']])
    CONF_THRESHOLD=float(chosen['confidence'])
    print('Chosen validation confidence threshold:',CONF_THRESHOLD,'| selection:',chosen)
    if TEST_IMAGE_DIR is not None:
        test_images=ml.list_images(TEST_IMAGE_DIR)
        TEST_CACHE=ml.cache_detection_predictions(detector,test_images,imgsz=IMGSZ,device=DEVICE,low_conf=.01)
        test_custom_metrics=ml.detection_metrics_from_cache(TEST_CACHE,TEST_IMAGE_DIR,TEST_LABEL_DIR,
                                                         confidence=CONF_THRESHOLD,iou_threshold=.50)
        print('Held-out test metrics at validation-selected threshold:',test_custom_metrics)
else:
    print('Confidence calibration skipped.')

## 13. Inspect predictions on held-out images

Look for missed small defects, false positives on texture, and boxes that are too large. A high mAP does not mean every defect is safe to pass.

In [ ]:
if TASK == 'detection' and detector is not None and TEST_IMAGE_DIR is not None:
    sample_paths=ml.list_images(TEST_IMAGE_DIR)[:6]
    results=detector.predict(source=[str(p) for p in sample_paths],imgsz=IMGSZ,
                           conf=CONF_THRESHOLD,device=DEVICE,verbose=False)
    fig,axes=plt.subplots(2,3,figsize=(14,8))
    for ax,result in zip(axes.ravel(),results):
        ax.imshow(cv2.cvtColor(result.plot(),cv2.COLOR_BGR2RGB)); ax.axis('off')
    plt.tight_layout(); plt.show()
else:
    print('No detector test predictions to display.')

## 14. Robustness stress test — controlled visual drift

Use a **small subset of test images** to measure how brightness, contrast, blur and noise affect detector performance. These are synthetic perturbations; they do not represent every factory drift condition. Labels stay unchanged because the perturbations are photometric, not geometric.

In [ ]:
def cache_transformed_predictions(model,image_paths,transform_kind,conf_floor=.01):
    cache=[]
    for p in image_paths:
        image=cv2.imread(str(p))
        if image is None: continue
        shifted=ml.perturb_image(image,transform_kind,seed=SEED)
        pred=model.predict(source=shifted,imgsz=IMGSZ,conf=conf_floor,device=DEVICE,verbose=False)[0]
        items=[]
        if pred.boxes is not None and len(pred.boxes):
            for box in pred.boxes:
                items.append((int(box.cls.item()),box.xyxy[0].detach().cpu().numpy().astype(float),float(box.conf.item())))
        cache.append({'image_path':Path(p),'predictions':items})
    return cache

if TASK == 'detection' and detector is not None and TEST_IMAGE_DIR is not None:
    stress_images=ml.list_images(TEST_IMAGE_DIR)[:min(50,len(ml.list_images(TEST_IMAGE_DIR)))]
    stress_rows=[]
    for condition in ['original','dark','bright','contrast_low','blur','noise']:
        if condition=='original':
            stress_cache=TEST_CACHE[:len(stress_images)] if TEST_CACHE is not None else ml.cache_detection_predictions(detector,stress_images,imgsz=IMGSZ,device=DEVICE,low_conf=.01)
        else:
            stress_cache=cache_transformed_predictions(detector,stress_images,condition)
        metric=ml.detection_metrics_from_cache(stress_cache,TEST_IMAGE_DIR,TEST_LABEL_DIR,CONF_THRESHOLD,.50)
        stress_rows.append({k:metric[k] for k in ['confidence','precision','recall','f1','tp','fp','fn'] if k in metric})
        stress_rows[-1]['condition']=condition
    display(pd.DataFrame(stress_rows)[['condition','confidence','precision','recall','f1','tp','fp','fn']])
else:
    print('Stress testing skipped.')

## 15. Tiled inference for tiny defects

For large camera frames, full-frame resizing can make very small defects disappear. Tiling increases the effective pixel size of a defect. It costs more inference time, so compare miss rate and latency before selecting it for the edge device.

In [ ]:
if TASK == 'detection' and detector is not None:
    candidates=ml.list_images(VAL_IMAGE_DIR if VAL_IMAGE_DIR is not None else Path(YOLO_SUMMARY['train']['image_dir']))
    if candidates:
        sample=cv2.imread(str(candidates[0]))
        tiled=ml.tiled_predict(detector,sample,class_names=CLASSES,tile_size=640,overlap=.20,
                              imgsz=IMGSZ,conf=CONF_THRESHOLD,device=DEVICE)
        print('Tiled predictions:',len(tiled),'(prototype; measure latency and NMS behavior)')
else:
    print('Tiled detection skipped.')

## 16. Optional image-classification branch

When the selected dataset has class folders but no real localization annotations, this branch trains MobileNetV3-Small with ImageNet transfer learning. It never invents bounding boxes. The default `gc10_hf` dataset remains on the detection branch.

In [ ]:
CLASSIFIER = None
CLASSIFIER_BEST = None
CLASSIFICATION_METRICS = None
CLASS_NAMES = None
if TASK == 'classification' and RUN_TRAINING:
    processed=ml.prepare_classification_splits(raw,CLASSIFICATION_ROOT,seed=SEED)
    corrupt_class_images=[]
    for image_path in ml.list_images(processed):
        try:
            with Image.open(image_path) as im: im.verify()
        except Exception: corrupt_class_images.append(str(image_path))
    if corrupt_class_images:
        raise RuntimeError(f'Unreadable classification images found; remove/fix them before training: {corrupt_class_images[:10]}')
    from torchvision import transforms, datasets
    from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
    from torch.utils.data import DataLoader
    from torch import nn
    from torch.optim import AdamW
    from sklearn.metrics import accuracy_score, f1_score
    IMAGE_SIZE=224
    train_transform=transforms.Compose([
        transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(.5),
        transforms.RandomRotation(8),
        transforms.ColorJitter(brightness=.12,contrast=.12,saturation=.08),
        transforms.ToTensor(),
        transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
    eval_transform=transforms.Compose([
        transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),transforms.ToTensor(),
        transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
    train_ds=datasets.ImageFolder(processed/'train',transform=train_transform)
    val_ds=datasets.ImageFolder(processed/'val',transform=eval_transform)
    test_ds=datasets.ImageFolder(processed/'test',transform=eval_transform)
    CLASS_NAMES=train_ds.classes
    assert train_ds.classes==val_ds.classes==test_ds.classes, 'Class mapping mismatch across splits.'
    class_counts=Counter(train_ds.targets)
    weights=torch.tensor([len(train_ds)/max(class_counts[i],1) for i in range(len(CLASS_NAMES))],dtype=torch.float32)
    weights=weights/weights.mean()
    criterion=nn.CrossEntropyLoss(weight=weights.to(DEVICE))
    try:
        CLASSIFIER=mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT)
        print('Loaded pretrained ImageNet weights.')
    except Exception as pretrained_error:
        warnings.warn(f'Pretrained weights unavailable ({pretrained_error}); training from scratch will likely be weaker.')
        CLASSIFIER=mobilenet_v3_small(weights=None)
    CLASSIFIER.classifier[-1]=nn.Linear(CLASSIFIER.classifier[-1].in_features,len(CLASS_NAMES))
    CLASSIFIER=CLASSIFIER.to(DEVICE)
    for p in CLASSIFIER.features.parameters(): p.requires_grad=False
    optimizer=AdamW(CLASSIFIER.classifier.parameters(),lr=1e-3,weight_decay=1e-4)
    train_loader=DataLoader(train_ds,batch_size=32,shuffle=True,num_workers=WORKERS,pin_memory=torch.cuda.is_available())
    val_loader=DataLoader(val_ds,batch_size=32,shuffle=False,num_workers=WORKERS,pin_memory=torch.cuda.is_available())
    test_loader=DataLoader(test_ds,batch_size=32,shuffle=False,num_workers=WORKERS,pin_memory=torch.cuda.is_available())
    CLASSIFICATION_EPOCHS=12
    best_val_loss=float('inf'); best_state=None; patience=4; stale=0; history=[]
    for epoch in range(CLASSIFICATION_EPOCHS):
        if epoch==2:
            for p in CLASSIFIER.features.parameters(): p.requires_grad=True
            optimizer=AdamW(CLASSIFIER.parameters(),lr=1e-4,weight_decay=1e-4)
        CLASSIFIER.train(); running_loss=0; train_correct=0; train_total=0
        for x,y in train_loader:
            x=x.to(DEVICE); y=y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits=CLASSIFIER(x); loss=criterion(logits,y); loss.backward(); optimizer.step()
            running_loss+=loss.item()*len(y); train_correct+=(logits.argmax(1)==y).sum().item(); train_total+=len(y)
        CLASSIFIER.eval(); val_loss=0; val_correct=0; val_total=0
        with torch.inference_mode():
            for x,y in val_loader:
                x=x.to(DEVICE); y=y.to(DEVICE); logits=CLASSIFIER(x)
                val_loss+=criterion(logits,y).item()*len(y); val_correct+=(logits.argmax(1)==y).sum().item(); val_total+=len(y)
        val_loss/=max(val_total,1); val_acc=val_correct/max(val_total,1)
        history.append({'epoch':epoch+1,'train_loss':running_loss/max(train_total,1),'train_acc':train_correct/max(train_total,1),'val_loss':val_loss,'val_acc':val_acc})
        print(f'Epoch {epoch+1:02d}/{CLASSIFICATION_EPOCHS} | train loss {history[-1]["train_loss"]:.4f} acc {history[-1]["train_acc"]:.3f} | val loss {val_loss:.4f} acc {val_acc:.3f}')
        if val_loss<best_val_loss:
            best_val_loss=val_loss; best_state={k:v.detach().cpu().clone() for k,v in CLASSIFIER.state_dict().items()}; stale=0
        else:
            stale+=1
            if stale>=patience:
                print('Early stopping triggered.'); break
    CLASSIFIER.load_state_dict(best_state); CLASSIFIER.eval()
    def classify_loader(loader):
        y_true=[]; y_pred=[]; probs=[]
        with torch.inference_mode():
            for x,y in loader:
                logits=CLASSIFIER(x.to(DEVICE)); p=torch.softmax(logits,dim=1).cpu()
                y_true.extend(y.tolist()); y_pred.extend(p.argmax(1).tolist()); probs.extend(p.tolist())
        return np.asarray(y_true),np.asarray(y_pred),np.asarray(probs)
    y_true,y_pred,y_prob=classify_loader(test_loader)
    CLASSIFICATION_METRICS={'accuracy':float(accuracy_score(y_true,y_pred)),'macro_f1':float(f1_score(y_true,y_pred,average='macro'))}
    print('Held-out classification test:',CLASSIFICATION_METRICS)
    print(classification_report(y_true,y_pred,labels=list(range(len(CLASS_NAMES))),target_names=CLASS_NAMES,zero_division=0))
    display(pd.DataFrame(confusion_matrix(y_true,y_pred,labels=list(range(len(CLASS_NAMES)))),index=CLASS_NAMES,columns=CLASS_NAMES))
    ax=pd.DataFrame(history).set_index('epoch')[['train_loss','val_loss']].plot(title='Classifier loss'); plt.show()
    CLASSIFIER_BEST=ROOT/'classification_model.pt'
    torch.save({'state_dict':CLASSIFIER.state_dict(),'classes':CLASS_NAMES,'image_size':IMAGE_SIZE},CLASSIFIER_BEST)
    # Temperature scaling: fit a single calibration temperature on validation logits.
    import torch.nn.functional as F
    def collect_logits(loader):
        logits_all=[]; labels_all=[]
        with torch.inference_mode():
            for xb,yb in loader:
                logits_all.append(CLASSIFIER(xb.to(DEVICE)).detach().cpu()); labels_all.append(yb.cpu())
        return torch.cat(logits_all),torch.cat(labels_all)
    val_logits,val_labels=collect_logits(val_loader); test_logits,test_labels=collect_logits(test_loader)
    log_temperature=nn.Parameter(torch.zeros(1,device=DEVICE))
    temp_optimizer=torch.optim.LBFGS([log_temperature],lr=.1,max_iter=50)
    val_logits_device=val_logits.to(DEVICE); val_labels_device=val_labels.to(DEVICE)
    def temperature_closure():
        temp_optimizer.zero_grad()
        temperature=log_temperature.exp().clamp(.05,10.0)
        loss=F.cross_entropy(val_logits_device/temperature,val_labels_device)
        loss.backward()
        return loss
    temp_optimizer.step(temperature_closure)
    CLASSIFIER_TEMPERATURE=float(log_temperature.detach().exp().clamp(.05,10.0).cpu().item())
    def calibrated_ece(logits,labels,temperature=1.0,bins=10):
        probabilities=torch.softmax(logits/temperature,dim=1).numpy(); labels=labels.numpy()
        confidence=probabilities.max(axis=1); predictions=probabilities.argmax(axis=1); correctness=(predictions==labels)
        ece=0.0
        edges=np.linspace(0.0,1.0,bins+1)
        for i in range(bins):
            if i<bins-1: mask=(confidence>=edges[i])&(confidence<edges[i+1])
            else: mask=(confidence>=edges[i])&(confidence<=edges[i+1])
            if mask.any(): ece+=mask.mean()*abs(correctness[mask].mean()-confidence[mask].mean())
        return float(ece)
    val_prob_cal=torch.softmax(val_logits/CLASSIFIER_TEMPERATURE,dim=1).numpy()
    val_conf=val_prob_cal.max(axis=1); val_pred=val_prob_cal.argmax(axis=1); val_truth=val_labels.numpy()
    CLASSIFIER_REVIEW_THRESHOLD=.95
    for threshold in np.arange(.50,.951,.05):
        accepted=val_conf>=threshold
        if accepted.any() and (val_pred[accepted]!=val_truth[accepted]).mean()<=.05:
            CLASSIFIER_REVIEW_THRESHOLD=float(round(threshold,2)); break
    test_prob_cal=torch.softmax(test_logits/CLASSIFIER_TEMPERATURE,dim=1).numpy()
    test_conf=test_prob_cal.max(axis=1); test_pred_cal=test_prob_cal.argmax(axis=1); test_truth=test_labels.numpy()
    accepted=test_conf>=CLASSIFIER_REVIEW_THRESHOLD
    CLASSIFICATION_METRICS.update({'temperature':CLASSIFIER_TEMPERATURE,'validation_ECE':calibrated_ece(val_logits,val_labels,CLASSIFIER_TEMPERATURE),
        'test_ECE':calibrated_ece(test_logits,test_labels,CLASSIFIER_TEMPERATURE),'review_threshold':CLASSIFIER_REVIEW_THRESHOLD,
        'test_review_coverage':float(accepted.mean()),'accepted_test_accuracy':float((test_pred_cal[accepted]==test_truth[accepted]).mean()) if accepted.any() else None})
    print('Calibrated classifier metrics and review policy:',CLASSIFICATION_METRICS)
    # Save calibration values alongside model weights for deployment.
    torch.save({'state_dict':CLASSIFIER.state_dict(),'classes':CLASS_NAMES,'image_size':IMAGE_SIZE,
               'temperature':CLASSIFIER_TEMPERATURE,'review_threshold':CLASSIFIER_REVIEW_THRESHOLD},CLASSIFIER_BEST)
elif TASK == 'classification':
    processed=ml.prepare_classification_splits(raw,CLASSIFICATION_ROOT,seed=SEED)
    print('Classification training is disabled. Set RUN_TRAINING=True to train.')
else:
    print('Classification training skipped: detection dataset selected.')

## 17. Optional anomaly branch — normal-only PatchCore-inspired memory

This branch is disabled by default. Enable `ANOMALY_MODE='mvtec_manual'` after extracting MVTec AD into `MVTEC_ROOT/<category>/...`, or `ANOMALY_MODE='custom'` and arrange your own images:

```text
custom_anomaly_data/
├── train/normal/       # normal-only feature memory
├── val/normal/        # normal-only threshold calibration
└── test/
    ├── good/
    └── defect/
```

**Never fuse an anomaly model with a detector from a different product/camera domain.** A tile memory bank is not a steel or PCB model.

In [ ]:
ANOMALY_READY=False
ANOMALY_PRODUCT=''
ANOMALY_TRAIN_IMAGES=[]; ANOMALY_CAL_IMAGES=[]; ANOMALY_TEST_GOOD=[]; ANOMALY_TEST_DEFECT=[]; ANOMALY_TEST_MASKS={}
if ANOMALY_MODE=='mvtec_manual':
    category_root=MVTEC_ROOT/MVTEC_CATEGORY
    train_dir=category_root/'train'/'good'
    test_dir=category_root/'test'
    if not train_dir.exists() or not test_dir.exists():
        raise FileNotFoundError(f'MVTec structure missing. Expected {train_dir} and {test_dir}; download/extract MVTec AD manually.')
    all_train_good=ml.list_images(train_dir)
    if len(all_train_good)<30: raise ValueError('Need at least 30 normal training images for this demo branch.')
    rng=random.Random(SEED); rng.shuffle(all_train_good); n_cal=max(5,int(.20*len(all_train_good)))
    ANOMALY_CAL_IMAGES=all_train_good[:n_cal]; ANOMALY_TRAIN_IMAGES=all_train_good[n_cal:]
    ANOMALY_TEST_GOOD=ml.list_images(test_dir/'good')
    for defect_dir in test_dir.iterdir():
        if defect_dir.is_dir() and defect_dir.name.lower()!='good': ANOMALY_TEST_DEFECT.extend(ml.list_images(defect_dir))
    ANOMALY_PRODUCT='mvtec_'+MVTEC_CATEGORY
    # Exercise the dataset manager's loader to confirm it sees test samples and masks.
    if dm.TORCH_AVAILABLE:
        mvtec_test=dm.MVTecADDataset(MVTEC_ROOT,MVTEC_CATEGORY,split='test',image_size=224)
        print('MVTec dataset manager loader samples:',len(mvtec_test))
elif ANOMALY_MODE=='custom':
    ANOMALY_TRAIN_IMAGES=ml.list_images(CUSTOM_ANOMALY_ROOT/'train'/'normal')
    ANOMALY_CAL_IMAGES=ml.list_images(CUSTOM_ANOMALY_ROOT/'val'/'normal')
    ANOMALY_TEST_GOOD=ml.list_images(CUSTOM_ANOMALY_ROOT/'test'/'good')
    ANOMALY_TEST_DEFECT=ml.list_images(CUSTOM_ANOMALY_ROOT/'test'/'defect')
    ANOMALY_PRODUCT=ANOMALY_PRODUCT_ID
    if min(len(ANOMALY_TRAIN_IMAGES),len(ANOMALY_CAL_IMAGES))<10:
        raise ValueError('Custom anomaly branch requires >=10 normal train and >=10 normal calibration images in its specified folders.')
elif ANOMALY_MODE!='none':
    raise ValueError("ANOMALY_MODE must be 'none', 'mvtec_manual', or 'custom'.")
print('Anomaly branch ready:',ANOMALY_MODE,'| train normals:',len(ANOMALY_TRAIN_IMAGES),'| calibration normals:',len(ANOMALY_CAL_IMAGES),'| test good:',len(ANOMALY_TEST_GOOD),'| test defect:',len(ANOMALY_TEST_DEFECT))

## 18. Fit anomaly memory bank and calibrate on normal-only images

The lightweight feature memory is **PatchCore-inspired**, not a full official PatchCore implementation. It uses ImageNet features, normalized local patch embeddings, nearest cosine distance, a small memory bank and a high-quantile normal threshold. The threshold is calibrated on a separate normal set.

In [ ]:
ANOMALY_ENCODER=None; MEMORY_BANK=None; ANOMALY_THRESHOLD=None; anomaly_encoder_onnx=ROOT/'anomaly_encoder.onnx'; memory_bank_path=ROOT/'anomaly_memory_bank.npz'
if ANOMALY_MODE!='none':
    import torch.nn as nn
    import torch.nn.functional as F
    from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
    from torchvision import transforms
    try:
        backbone=mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT).eval()
        print('Anomaly branch using pretrained MobileNetV3-Small features.')
    except Exception as pretrained_error:
        warnings.warn(f'Could not load pretrained anomaly features: {pretrained_error}. Using random weights is not recommended for performance.')
        backbone=mobilenet_v3_small(weights=None).eval()
    ANOMALY_ENCODER=nn.Sequential(*list(backbone.features.children())[:8]).eval().to(DEVICE)
    anomaly_preprocess=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
    def anomaly_patch_features(image_path):
        im=Image.open(image_path).convert('RGB'); x=anomaly_preprocess(im).unsqueeze(0).to(DEVICE)
        with torch.inference_mode(): feature_map=ANOMALY_ENCODER(x).float().cpu()[0]
        c,h,w=feature_map.shape
        desc=feature_map.permute(1,2,0).reshape(-1,c)
        desc=F.normalize(desc,p=2,dim=1)
        return desc.numpy().astype(np.float32),(h,w)
    patch_list=[]
    for p in ANOMALY_TRAIN_IMAGES:
        features,_=anomaly_patch_features(p); patch_list.append(features)
    all_patches=np.concatenate(patch_list,axis=0)
    rng=np.random.default_rng(SEED)
    bank_size=min(ANOMALY_MEMORY_MAX,len(all_patches))
    MEMORY_BANK=all_patches[rng.choice(len(all_patches),size=bank_size,replace=False)].astype(np.float32)
    del all_patches,patch_list; gc.collect()
    def anomaly_score_path(image_path):
        desc,(h,w)=anomaly_patch_features(image_path)
        distances=1.0-(desc@MEMORY_BANK.T).max(axis=1)
        k=max(1,int(math.ceil(.05*len(distances))))
        score=float(np.mean(np.partition(distances,-k)[-k:]))
        heat=distances.reshape(h,w)
        return score,heat
    calibration_scores=np.array([anomaly_score_path(p)[0] for p in ANOMALY_CAL_IMAGES])
    ANOMALY_THRESHOLD=float(np.quantile(calibration_scores,ANOMALY_CALIBRATION_QUANTILE))
    np.savez_compressed(memory_bank_path,bank=MEMORY_BANK,threshold=np.array(ANOMALY_THRESHOLD),domain=np.array(ANOMALY_PRODUCT))
    print('Memory bank shape:',MEMORY_BANK.shape,'| Normal-calibrated threshold:',ANOMALY_THRESHOLD)
else:
    print('Anomaly branch disabled; no memory bank created.')

## 19. Anomaly evaluation and heatmap

Only image-level AUROC and test false-alarm/sensitivity are reported here. If MVTec masks are available, pixel-level evaluation can be added separately; this simple branch does not claim segmentation accuracy.

In [ ]:
ANOMALY_TEST_METRICS=None
if ANOMALY_MODE!='none':
    y_true=[]; y_scores=[]
    for p in ANOMALY_TEST_GOOD: y_true.append(0); y_scores.append(anomaly_score_path(p)[0])
    for p in ANOMALY_TEST_DEFECT: y_true.append(1); y_scores.append(anomaly_score_path(p)[0])
    if len(set(y_true))==2:
        auc=float(roc_auc_score(y_true,y_scores))
        flags=np.array(y_scores)>ANOMALY_THRESHOLD
        y_true_arr=np.array(y_true)
        fp=int(np.sum((y_true_arr==0)&flags)); tn=int(np.sum((y_true_arr==0)&(~flags)))
        tp=int(np.sum((y_true_arr==1)&flags)); fn=int(np.sum((y_true_arr==1)&(~flags)))
        ANOMALY_TEST_METRICS={'image_AUROC':auc,'normal_FPR':fp/max(fp+tn,1),'defect_sensitivity':tp/max(tp+fn,1),'test_good':len(ANOMALY_TEST_GOOD),'test_defect':len(ANOMALY_TEST_DEFECT)}
        print('Anomaly test metrics:',ANOMALY_TEST_METRICS)
    else:
        print('Both normal and defect test images are required to calculate AUROC.')
    heat_sample=(ANOMALY_TEST_DEFECT or ANOMALY_TEST_GOOD)
    if heat_sample:
        sample_path=heat_sample[0]; score,heat=anomaly_score_path(sample_path)
        image=Image.open(sample_path).convert('RGB').resize((224,224))
        heat_resized=cv2.resize(heat,(224,224),interpolation=cv2.INTER_CUBIC)
        fig,axes=plt.subplots(1,2,figsize=(10,4))
        axes[0].imshow(image); axes[0].set_title(f'Image: {Path(sample_path).name}'); axes[0].axis('off')
        axes[1].imshow(image); axes[1].imshow(heat_resized,alpha=.45); axes[1].set_title(f'Anomaly heatmap score={score:.4f}'); axes[1].axis('off')
        plt.tight_layout(); plt.show()
else:
    print('No anomaly evaluation run.')

## 20. Drift monitoring — simple image statistics

Monitor brightness, contrast and sharpness by product, camera and batch. This detects obvious input shifts, not all semantic or defect-distribution drift. **Do not auto-update the model or memory bank from drifted production data.** Use drift to trigger sampling and QA review.

In [ ]:
from scipy.stats import ks_2samp
DRIFT_TABLE=None
if TASK=='detection' and VAL_IMAGE_DIR is not None:
    baseline_paths=ml.list_images(VAL_IMAGE_DIR)
elif TASK=='classification' and 'processed' in globals():
    baseline_paths=ml.list_images(processed/'val')
else:
    baseline_paths=[]
if baseline_paths:
    baseline_stats=pd.DataFrame([ml.image_quality_stats(cv2.imread(str(p))) for p in baseline_paths[:100]])
    drift_rows=[]
    subset=baseline_paths[:min(40,len(baseline_paths))]
    for shift in ['dark','bright','contrast_low','blur','noise']:
        shifted_stats=pd.DataFrame([ml.image_quality_stats(ml.perturb_image(cv2.imread(str(p)),shift,seed=SEED)) for p in subset])
        for metric in baseline_stats.columns:
            stat,pvalue=ks_2samp(baseline_stats[metric].dropna(),shifted_stats[metric].dropna())
            drift_rows.append({'synthetic_shift':shift,'feature':metric,'reference_mean':baseline_stats[metric].mean(),'shifted_mean':shifted_stats[metric].mean(),'KS_statistic':stat,'p_value_exploratory':pvalue})
    DRIFT_TABLE=pd.DataFrame(drift_rows)
    display(DRIFT_TABLE)
else:
    print('No baseline images available for drift demonstration.')

## 21. Severity policy — rules are a prototype, not a trained severity model

Severity must be labeled by manufacturing experts. This notebook does not train a severity classifier because the registered datasets do not provide consistent minor/major/critical labels. The rule below is only a demonstration of how measured defect area can be translated into a candidate level.

In [ ]:
def severity_candidate(area_ratio, defect_name='unknown', critical_classes=None):
    critical_classes=set(critical_classes or [])
    if defect_name in critical_classes: return 'critical_candidate'
    if area_ratio>=0.10: return 'major_candidate'
    if area_ratio>=0.02: return 'minor_candidate'
    return 'review_candidate'

print('Example only: 2% defect-area ratio ->',severity_candidate(.02,'scratch'))
print('Replace these thresholds with expert-approved measurements and a severity-labelled dataset before use.')

## 22. Local SOP + Llama advisory payload

Llama is not a vision model and must not override inspection status. It receives structured detection/anomaly facts and a locally approved SOP snippet. On the Raspberry Pi, the advisory process uses the ExecuTorch `llama_main` runner and `.pte` model on the Arm CPU; vision may separately use CPU ONNX Runtime or HailoRT.

In [ ]:
SOP_MAP={
    'scratch':'Inspect handling contact points, rollers, fixtures and transport surfaces; compare adjacent components from the same batch.',
    'crazing':'Hold for magnified inspection; check material stress and process conditions.',
    'crack':'Hold and inspect crack depth/position; review the process and adjacent components.',
    'inclusion':'Hold for review; inspect material quality and surface cleanliness.',
    'pitted_surface':'Review surface finishing and corrosion-prevention process.',
    'unknown_anomaly':'Isolate the component and inspect the highlighted region manually.',
    'image_quality':'Recapture after correcting focus, exposure, lighting or camera stability.'}

def build_llama_payload(event):
    return {'role':'offline_quality_control_advisor','decision_is_locked':event.get('decision'),'facts':event,
            'approved_sop':event.get('sop_recommendation','No SOP provided.'),
            'instructions':['Use only the supplied facts and SOP.','Do not invent causes.','Do not change ACCEPT/HOLD/REVIEW/RECAPTURE decision.','No autonomous machine-control instructions.','Return a concise operator checklist and urgency.']}

sample_event={'component_id':'demo-001','product_id':DATA_MODE,'decision':'REVIEW','defects':[], 'anomaly_score':None,'quality':{'passed':True,'flags':[]},'sop_recommendation':SOP_MAP['unknown_anomaly']}
llama_prompt=json.dumps(build_llama_payload(sample_event),indent=2)
(ROOT/'sample_llama_prompt.json').write_text(llama_prompt,encoding='utf-8')
print(llama_prompt)

## 23. Export models and inference artifacts

Export the detector to ONNX for CPU inference or a separate Hailo compilation workflow. ONNX is **not** a Hailo HEF. The Llama model is not exported from this notebook; follow the ExecuTorch Raspberry Pi build/export guide.

In [ ]:
ARTIFACTS=[]
if RUN_EXPORT:
    if TASK=='detection' and detector is not None:
        if BEST_WEIGHTS is not None and Path(BEST_WEIGHTS).is_file():
            shutil.copy2(BEST_WEIGHTS,ROOT/'yolo_detector_best.pt')
            ARTIFACTS.append(ROOT/'yolo_detector_best.pt')
        try:
            yolo_onnx=Path(detector.export(format='onnx',imgsz=IMGSZ,opset=12,simplify=False,dynamic=False))
            stable_onnx=ROOT/'yolo_detector.onnx'
            shutil.copy2(yolo_onnx,stable_onnx)
            ARTIFACTS.append(stable_onnx)
            print('YOLO ONNX exported:',stable_onnx)
        except Exception as export_error:
            warnings.warn(f'YOLO ONNX export did not finish; the PyTorch checkpoint is retained if it was copied: {export_error}')
    if TASK=='classification' and CLASSIFIER is not None:
        try:
            import torch.nn as nn
            class ExportClassifier(nn.Module):
                def __init__(self,model): super().__init__(); self.model=model
                def forward(self,x): return self.model(x)
            export_model=ExportClassifier(CLASSIFIER.cpu().eval())
            torch.onnx.export(export_model,torch.randn(1,3,224,224),str(ROOT/'classifier.onnx'),
                              opset_version=17,input_names=['image'],output_names=['logits'],dynamo=False)
            ARTIFACTS.append(ROOT/'classifier.onnx'); CLASSIFIER.to(DEVICE)
            (ROOT/'classification_classes.json').write_text(json.dumps(CLASS_NAMES,indent=2),encoding='utf-8')
            print('Classifier ONNX exported.')
        except Exception as export_error:
            warnings.warn(f'Classifier ONNX export failed; PyTorch checkpoint is retained: {export_error}')
    if ANOMALY_MODE!='none' and ANOMALY_ENCODER is not None:
        try:
            ANOMALY_ENCODER.cpu().eval()
            torch.onnx.export(ANOMALY_ENCODER,torch.randn(1,3,224,224),str(anomaly_encoder_onnx),
                            opset_version=17,input_names=['image'],output_names=['features'],dynamo=False)
            ARTIFACTS.extend([anomaly_encoder_onnx,memory_bank_path])
            ANOMALY_ENCODER.to(DEVICE)
            print('Anomaly encoder ONNX exported.')
        except Exception as export_error:
            warnings.warn(f'Anomaly ONNX export failed: {export_error}')
else:
    print('Model export disabled.')

## 24. Create a manifest for reproducibility

The manifest records dataset, classes, thresholds, model filenames and limitations. It does not mark the system factory-certified.

In [ ]:
manifest={
    'project':'Industry4_Offline_Quality_Inspector','seed':SEED,'task':TASK,'dataset_mode':DATA_MODE,
    'dataset_id':dm.DATASETS.get(DATA_MODE,{}).get('id'),'classes':CLASSES if TASK=='detection' else CLASS_NAMES,
    'detector_model':MODEL_NAME if TASK=='detection' else None,'detector_weights':str(BEST_WEIGHTS) if BEST_WEIGHTS else None,
    'confidence_threshold_selected_on_validation':CONF_THRESHOLD if TASK=='detection' and detector is not None else None,
    'anomaly_model':'MobileNetV3 patch-memory (PatchCore-inspired)' if ANOMALY_MODE!='none' else None,
    'anomaly_product_domain':ANOMALY_PRODUCT if ANOMALY_MODE!='none' else None,
    'anomaly_threshold':ANOMALY_THRESHOLD if ANOMALY_MODE!='none' else None,
    'classification_metrics':CLASSIFICATION_METRICS,
    'anomaly_test_metrics':ANOMALY_TEST_METRICS,
    'factory_certified':False,
    'limitations':['Public benchmark data does not demonstrate production drift robustness.','Severity thresholds require expert approval.','Anomaly model must match the target product and acquisition setup.','Use batch/shift/camera-held-out validation for factory data.']}
(ROOT/'manifest.json').write_text(json.dumps(manifest,indent=2,default=str),encoding='utf-8')
print(json.dumps(manifest,indent=2,default=str))

## 25. Raspberry Pi offline inference application

This reference application uses YOLO weights (`.pt` or exported `.onnx`) and SQLite logging. Optional anomaly encoder + memory bank must have a matching product domain. Llama advice is event-triggered and does not control machinery.

In [ ]:
%%writefile /content/industry4_qc/edge_inspector.py
#!/usr/bin/env python3
"""Offline Raspberry Pi reference inspector: YOLO, optional anomaly memory, SQLite, optional Llama."""
from __future__ import annotations
import argparse, concurrent.futures, hashlib, json, sqlite3, subprocess, time, uuid
from datetime import datetime, timezone
from pathlib import Path
import cv2, numpy as np
from ultralytics import YOLO

SOPS = {
    "crazing": "Hold for magnified inspection; review material stress and process conditions.",
    "crack": "Hold and inspect crack depth/position; review process and adjacent components.",
    "inclusion": "Hold for review; inspect material quality and surface cleanliness.",
    "pitted_surface": "Review surface finishing and corrosion-prevention process.",
    "scratch": "Inspect handling contact points, rollers, fixtures and transport surfaces.",
    "unknown_anomaly": "Isolate the component and inspect the highlighted region manually.",
    "image_quality": "Recapture after correcting focus, exposure, lighting or camera stability.",
    "default": "Hold for operator review; verify the defect and compare the production batch.",
}

def image_quality(frame, dark_min=20.0, bright_max=245.0, blur_min=15.0):
    gray=cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)
    brightness=float(gray.mean())
    sharpness=float(cv2.Laplacian(gray,cv2.CV_64F).var())
    flags=[]
    if brightness<dark_min: flags.append("underexposed")
    if brightness>bright_max: flags.append("overexposed")
    if sharpness<blur_min: flags.append("blurry")
    return {"brightness":brightness,"sharpness":sharpness,"flags":flags,"passed":not flags}

def open_db(path):
    db=sqlite3.connect(path)
    db.execute("""CREATE TABLE IF NOT EXISTS inspections(
        event_id TEXT PRIMARY KEY, timestamp_utc TEXT, product_id TEXT,
        decision TEXT, defects_json TEXT, anomaly_score REAL, anomaly_alarm INTEGER,
        quality_json TEXT, model_sha TEXT, latency_ms REAL, sop TEXT, llama_output TEXT,
        operator_feedback TEXT)""")
    db.commit()
    # Add the feedback column when upgrading a database created by an older app version.
    existing_columns={row[1] for row in db.execute("PRAGMA table_info(inspections)")}
    if "operator_feedback" not in existing_columns:
        db.execute("ALTER TABLE inspections ADD COLUMN operator_feedback TEXT")
        db.commit()
    return db

def file_sha256(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""):
            h.update(chunk)
    return h.hexdigest()[:16]

def run_llama_advisory(runner, model_path, tokenizer_path, event, sop, timeout=120):
    payload={"role":"quality_control_advisor","locked_decision":event["decision"],
             "inspection_facts":event,"approved_sop":sop,
             "instructions":["Do not change the inspection decision.","Use only supplied facts and approved SOP.","Do not invent causes.","Give a short checklist; no machine actuation."]}
    prompt=json.dumps(payload,separators=(",",":"))
    command=[str(runner),"--model_path",str(model_path),"--tokenizer_path",str(tokenizer_path),
            "--seq_len","256","--prompt",prompt]
    result=subprocess.run(command,capture_output=True,text=True,timeout=timeout,check=False)
    if result.returncode!=0:
        return "EXECUTORCH_ERROR: "+(result.stderr or result.stdout)[-1200:]
    return result.stdout.strip()[-3000:]

class OptionalAnomaly:
    """ONNX MobileNet patch encoder + cosine memory bank for same-domain anomaly checks."""
    def __init__(self, encoder_path, bank_path, expected_domain):
        import onnxruntime as ort
        self.session=ort.InferenceSession(str(encoder_path),providers=["CPUExecutionProvider"])
        archive=np.load(bank_path,allow_pickle=False)
        self.bank=archive["bank"].astype(np.float32)
        self.threshold=float(archive["threshold"])
        self.domain=str(archive["domain"].item())
        if self.domain!=expected_domain:
            raise ValueError(f"Anomaly bank domain {self.domain!r} does not match requested product {expected_domain!r}.")
        self.input_name=self.session.get_inputs()[0].name
    def score(self, frame_bgr):
        image=cv2.cvtColor(frame_bgr,cv2.COLOR_BGR2RGB)
        image=cv2.resize(image,(224,224),interpolation=cv2.INTER_AREA).astype(np.float32)/255.0
        mean=np.array([.485,.456,.406],np.float32); std=np.array([.229,.224,.225],np.float32)
        tensor=((image-mean)/std).transpose(2,0,1)[None].astype(np.float32)
        feature=self.session.run(None,{self.input_name:tensor})[0][0]
        c,h,w=feature.shape
        desc=feature.transpose(1,2,0).reshape(-1,c)
        desc=desc/np.maximum(np.linalg.norm(desc,axis=1,keepdims=True),1e-8)
        distances=1.0-(desc@self.bank.T).max(axis=1)
        k=max(1,int(np.ceil(.05*len(distances))))
        score=float(np.mean(np.partition(distances,-k)[-k:]))
        heat=distances.reshape(h,w)
        return score,heat

def main():
    parser=argparse.ArgumentParser(description="Offline quality inspection app (human-reviewed candidate decisions)")
    parser.add_argument("--weights",help="YOLO .pt or ONNX model; optional only with --disable-detector")
    parser.add_argument("--detector-product",default="steel_gc10")
    parser.add_argument("--disable-detector",action="store_true")
    parser.add_argument("--product",default="steel_gc10")
    parser.add_argument("--source",default="0",help="camera index, image file, or video file")
    parser.add_argument("--db",default="inspections.sqlite3")
    parser.add_argument("--conf",type=float,default=.25)
    parser.add_argument("--imgsz",type=int,default=640)
    parser.add_argument("--interval",type=float,default=.5)
    parser.add_argument("--view",action="store_true")
    parser.add_argument("--evidence-dir",type=Path)
    parser.add_argument("--anomaly-encoder",type=Path)
    parser.add_argument("--anomaly-bank",type=Path)
    parser.add_argument("--llama-runner",type=Path)
    parser.add_argument("--llama-model",type=Path)
    parser.add_argument("--llama-tokenizer",type=Path)
    args=parser.parse_args()

    detector=None; model_sha="none"
    if not args.disable_detector:
        if not args.weights: parser.error("Provide --weights or use --disable-detector")
        if args.product!=args.detector_product:
            raise ValueError("Product identifier does not match detector training domain; refusing to run.")
        weights=Path(args.weights)
        if not weights.is_file(): raise FileNotFoundError(weights)
        detector=YOLO(str(weights),task="detect"); model_sha=file_sha256(weights)
    anomaly=None
    if bool(args.anomaly_encoder)!=bool(args.anomaly_bank):
        parser.error("Provide both --anomaly-encoder and --anomaly-bank, or neither.")
    if args.anomaly_encoder and args.anomaly_bank:
        anomaly=OptionalAnomaly(args.anomaly_encoder,args.anomaly_bank,expected_domain=args.product)

    db=open_db(args.db)
    if args.evidence_dir: args.evidence_dir.mkdir(parents=True,exist_ok=True)
    llama_paths=(args.llama_runner,args.llama_model,args.llama_tokenizer)
    llama_ready=all(x is not None and x.is_file() for x in llama_paths)
    if any(x is not None for x in llama_paths) and not llama_ready:
        print("Warning: Llama advisory paths are incomplete; advisory inference is disabled.")
    pool=concurrent.futures.ThreadPoolExecutor(max_workers=1) if llama_ready else None
    pending=None
    source=int(args.source) if str(args.source).isdigit() else str(args.source)
    image_exts={".jpg",".jpeg",".png",".bmp",".webp"}
    single_image=isinstance(source,str) and Path(source).suffix.lower() in image_exts
    capture=None if single_image else cv2.VideoCapture(source)
    if not single_image and not capture.isOpened():
        db.close();
        raise RuntimeError(f"Camera/video unavailable: {source}")
    last_time=-1e9
    try:
        while True:
            if single_image:
                frame=cv2.imread(source)
                if frame is None: raise RuntimeError(f"Cannot read image: {source}")
            else:
                ok,frame=capture.read()
                if not ok: break
            if not single_image and time.monotonic()-last_time<args.interval:
                time.sleep(0.005)
                continue
            last_time=time.monotonic(); start=time.perf_counter(); quality=image_quality(frame)
            result=None; defects=[]; anomaly_score=None; anomaly_alarm=False; heatmap=None
            if quality["passed"]:
                if detector is not None:
                    result=detector.predict(frame,conf=args.conf,imgsz=args.imgsz,device="cpu",verbose=False)[0]
                    for box in result.boxes:
                        cls_id=int(box.cls.item()); x1,y1,x2,y2=[float(v) for v in box.xyxy[0].tolist()]
                        height,width=frame.shape[:2]
                        area_ratio=max(0.0,x2-x1)*max(0.0,y2-y1)/(height*width)
                        defects.append({"class_id":cls_id,"class_name":detector.names[cls_id],
                                      "confidence":round(float(box.conf.item()),4),
                                      "bbox_xyxy":[round(x1,1),round(y1,1),round(x2,1),round(y2,1)],
                                      "area_ratio":round(area_ratio,6)})
                if anomaly is not None:
                    anomaly_score,heatmap=anomaly.score(frame)
                    anomaly_alarm=anomaly_score>anomaly.threshold
            if not quality["passed"]:
                decision="RECAPTURE"; sop=SOPS["image_quality"]
            elif defects:
                decision="HOLD"; sop=SOPS.get(str(defects[0]["class_name"]).lower(),SOPS["default"])
            elif anomaly_alarm:
                decision="REVIEW"; sop=SOPS["unknown_anomaly"]
            elif anomaly is not None:
                decision="ACCEPT_CANDIDATE"; sop="No anomaly signal; continue periodic QA audits."
            else:
                decision="REVIEW_UNVERIFIED"; sop="No product-specific anomaly model is loaded; do not automatically pass clean items."
            event_id=str(uuid.uuid4()); timestamp=datetime.now(timezone.utc).isoformat()
            latency_ms=(time.perf_counter()-start)*1000
            event={"event_id":event_id,"timestamp_utc":timestamp,"product_id":args.product,
                   "decision":decision,"quality":quality,"defects":defects,"anomaly_score":anomaly_score,
                   "anomaly_alarm":bool(anomaly_alarm),"model_sha":model_sha,"latency_ms":round(latency_ms,2)}
            db.execute("INSERT INTO inspections(event_id,timestamp_utc,product_id,decision,defects_json,anomaly_score,anomaly_alarm,quality_json,model_sha,latency_ms,sop,llama_output,operator_feedback) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)",
                    (event_id,timestamp,args.product,decision,json.dumps(defects),anomaly_score,int(anomaly_alarm),
                     json.dumps(quality),model_sha,latency_ms,sop,None,None)); db.commit()
            print(json.dumps({**event,"sop":sop},default=str))
            if pool and decision!="ACCEPT_CANDIDATE" and (pending is None or pending.done()):
                def worker(eid,payload,sop_text):
                    try: output=run_llama_advisory(args.llama_runner,args.llama_model,args.llama_tokenizer,payload,sop_text)
                    except Exception as exc: output="EXECUTORCH_ERROR: "+str(exc)
                    with sqlite3.connect(args.db) as worker_db:
                        worker_db.execute("UPDATE inspections SET llama_output=? WHERE event_id=?",(output,eid)); worker_db.commit()
                    print("Llama advisory (not decision input): ",output[:500])
                pending=pool.submit(worker,event_id,event,sop)
            if args.evidence_dir: cv2.imwrite(str(args.evidence_dir/f"{event_id}.jpg"),frame)
            if args.view:
                shown=result.plot() if result is not None else frame.copy()
                cv2.putText(shown,decision,(8,28),cv2.FONT_HERSHEY_SIMPLEX,.8,(0,0,255),2)
                if heatmap is not None:
                    hm=cv2.resize(heatmap.astype(np.float32),(frame.shape[1],frame.shape[0]))
                    hm=cv2.normalize(hm,None,0,255,cv2.NORM_MINMAX).astype(np.uint8)
                    overlay=cv2.applyColorMap(hm,cv2.COLORMAP_JET)
                    shown=cv2.addWeighted(shown,.7,overlay,.3,0)
                cv2.imshow("Offline QC Inspector (q=quit)",shown)
                if cv2.waitKey(1)&0xff==ord("q"): break
            if single_image: break
    finally:
        if capture is not None: capture.release()
        if args.view: cv2.destroyAllWindows()
        if pool: pool.shutdown(wait=True)
        db.close()

if __name__=="__main__":
    main()


## 26. Operator-feedback tool (does not retrain automatically)

Feedback is recorded for a human-reviewed update cycle. Do not auto-add false alarms to the normal memory bank without QA approval.

In [ ]:
%%writefile /content/industry4_qc/operator_feedback.py
#!/usr/bin/env python3
"""Record/export human review labels; does not retrain or update models automatically."""
import argparse
import csv
import sqlite3

VALID = {"confirmed_defect", "false_alarm", "missed_defect", "confirmed_good", "uncertain"}

def main():
    parser = argparse.ArgumentParser(description="Record or export human QC feedback")
    parser.add_argument("--db", default="inspections.sqlite3")
    parser.add_argument("--list", action="store_true")
    parser.add_argument("--event-id")
    parser.add_argument("--verdict", choices=sorted(VALID))
    parser.add_argument("--export")
    args = parser.parse_args()
    db = sqlite3.connect(args.db)
    columns = {row[1] for row in db.execute("PRAGMA table_info(inspections)")}
    if columns and "operator_feedback" not in columns:
        db.execute("ALTER TABLE inspections ADD COLUMN operator_feedback TEXT")
        db.commit()
    if args.list:
        for row in db.execute("SELECT event_id,timestamp_utc,product_id,decision,defects_json FROM inspections ORDER BY timestamp_utc DESC LIMIT 20"):
            print(row)
    if args.event_id or args.verdict:
        if not (args.event_id and args.verdict):
            parser.error("Provide --event-id and --verdict together")
        if db.execute("SELECT 1 FROM inspections WHERE event_id=?", (args.event_id,)).fetchone() is None:
            parser.error("Unknown event ID")
        db.execute("UPDATE inspections SET operator_feedback=? WHERE event_id=?", (args.verdict, args.event_id))
        db.commit()
        print("Feedback recorded; model is unchanged.")
    if args.export:
        rows = list(db.execute("SELECT event_id,timestamp_utc,product_id,decision,defects_json,operator_feedback FROM inspections WHERE operator_feedback IS NOT NULL"))
        with open(args.export, "w", newline="", encoding="utf-8") as handle:
            writer = csv.writer(handle)
            writer.writerow(["event_id","timestamp_utc","product_id","decision","defects_json","operator_feedback"])
            writer.writerows(rows)
        print(f"Exported {len(rows)} feedback rows to {args.export}")
    db.close()

if __name__ == "__main__":
    main()


## 27. Build the offline deployment bundle

The bundle contains model artifacts that were successfully exported, the manifest, the offline camera/inspection app, and the feedback tool. It **does not contain Llama weights or the ExecuTorch runner**; these must be built/downloaded separately on the supported host and transferred to the Pi.

In [ ]:
package_path=ROOT/'edge_deployment_bundle.zip'
files_to_package=['manifest.json','edge_inspector.py','operator_feedback.py','sample_llama_prompt.json',
                 'dataset_manager.py','industry4_ml_utils.py','yolo_detector_best.pt','yolo_detector.onnx',
                 'classification_model.pt','classifier.onnx','classification_classes.json','anomaly_encoder.onnx','anomaly_memory_bank.npz']
with zipfile.ZipFile(package_path,'w',zipfile.ZIP_DEFLATED) as archive:
    for name in files_to_package:
        path=ROOT/name
        if path.is_file(): archive.write(path,arcname=name)
print('Bundle created:',package_path,'| size MB:',round(package_path.stat().st_size/1e6,2))
try:
    from google.colab import files
    files.download(str(package_path))
except Exception:
    print('Not running in Colab; bundle is saved at:',package_path)

## 28. Raspberry Pi deployment notes

### Vision
- **CPU-only:** run the exported detector with Ultralytics/ONNX Runtime, or the PyTorch checkpoint if performance allows.
- **AI HAT / Hailo:** convert the trained vision model into the device-specific HEF using the supported Hailo toolchain. HailoRT runs that HEF on the NPU. This compilation is separate from Colab export.

### Llama advisory
- Build the ExecuTorch Llama runner on a supported Linux x86-64 host for the Pi target, and prepare the Llama 3.2 1B `.pte` plus `tokenizer.model`.
- The Raspberry Pi CPU runs `llama_main` for **event-triggered advisory only**. The decision engine and camera/vision pipeline remain separate.
- Current runner invocation follows the ExecuTorch Raspberry Pi tutorial:

```bash
./llama_main --model_path ./llama3_2.pte --tokenizer_path ./tokenizer.model --seq_len 256 --prompt "Summarize this quality-control event without changing its decision."
```

### Example camera run

```bash
python edge_inspector.py --weights yolo_detector_best.pt --product steel_gc10 --detector-product steel_gc10 --source 0 --view --db inspections.sqlite3
```

For an image:

```bash
python edge_inspector.py --weights yolo_detector_best.pt --product steel_gc10 --detector-product steel_gc10 --source /path/to/component.jpg
```

For optional Llama, add the three paths: `--llama-runner`, `--llama-model`, `--llama-tokenizer`. Advisory output must never be wired to machine-control decisions.

## 29. Recommended experiments to increase accuracy

Run these as controlled comparisons. Change one major factor at a time and retain the same batch-held-out test split.

1. **Data quality:** remove corrupt images; inspect class/box labels; hash and remove exact duplicates; keep related frames/components in the same split.
2. **Model size:** compare `yolo11n.pt` vs `yolo11s.pt`; use the best accuracy/latency trade-off, not the largest model by default.
3. **Resolution:** compare `IMGSZ=640` vs `960`; defects too small after resize benefit from tiling or close-up acquisition.
4. **Augmentation:** conservative brightness/contrast/noise/blur; do not use transforms that erase or distort tiny defects beyond real camera variability.
5. **Confidence threshold:** tune only on validation; choose a recall target suitable for QC, then measure test FP rate and FN.
6. **Hard-negative mining:** review false alarms from training/validation and approved production feedback, label them, and retrain. Never use test images for training.
7. **Anomaly model:** collect normal samples from the same product, camera, lighting and batch conditions; compare anomaly recall and normal false-alarm rate.
8. **Domain drift:** log brightness, sharpness, camera exposure, batch/shift, confidence distribution and operator overrides. Drift triggers QA review, not automatic adaptation.
9. **Small defects:** fixed optics, stable lighting, ROI alignment, high-resolution crops and tile inference often beat architectural changes.
10. **Edge constraints:** benchmark p50/p95 end-to-end latency, throughput, RAM, CPU temperature/throttling and false alarms per 1,000 normal units.

## 30. Sources and runtime separation

- GC10-DET Hugging Face: https://huggingface.co/datasets/dronefreak/GC10-DET
- NEU Surface Defect Database: https://www.kaggle.com/datasets/kaustubhdikshit/neu-surface-defect-database
- MVTec AD: https://www.mvtec.com/company/research/datasets/mvtec-ad
- Ultralytics validation: https://docs.ultralytics.com/modes/val
- Hailo integration: https://docs.ultralytics.com/integrations/hailo/
- ExecuTorch Raspberry Pi Llama: https://docs.pytorch.org/executorch/stable/raspberry_pi_llama_tutorial.html

**Separate pipelines:** `YOLO/ONNX → CPU runtime` OR `YOLO → Hailo compiler → HEF → HailoRT`; `Llama 3.2 → ExecuTorch `.pte` → Arm CPU`. Never describe the Hailo NPU as running Llama.